In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


In [2]:
# ===== NEW CELL — run first, immediately after the restart from cell 0 =====
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"])

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
%%writefile model.py
##################################################
# Defines HELM Phase 13A: Elastic Threshold Router
# Has a total of 32 heads, d_head = 64; only 16 will be used at a time
# True Decoupling of d_model = d_head * n_head
# Acheived via expansion layer
# Target 8 16 32
# 4 Perm heads
# No Dead Head Penalty
# Still Use Clamp
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm"

    def __init__(
        self,
        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 32,
        d_head = 64,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # Phase 13A-v2 Router Geometry
        num_router_latents = 4,
        num_permanent_heads = 2,
        head_target_min = 8,       # hard law: total heads can never fall below this
        head_target_center = 16,   # long-run controller target; NOT a per-example target
        head_target_max = 32,      # hard law / structural ceiling
        router_grad_clip = 0.05,

        # Router warmup schedule (fractions of dataset_total_steps)
        # A: all heads execute, router learns only through the CE/STE path.
        dense_router_warmup = 0.03,
        # B: exactly head_target_center heads execute. Per-head thresholds are calibrated
        #    so one fixed subset cannot permanently monopolize the layer.
        calibration_router_warmup = 0.03,
        # C transition: elastic routing is live; margin + controllers ramp in smoothly.
        elastic_transition = 0.015,

        # STE + confidence margin
        ste_temperature = 0.5,     # fixed: never learned, so scale cannot be gamed
        margin_delta = 0.15,       # desired signed distance from ON/OFF boundary
        margin_lambda = 0.001,     # final coefficient after elastic transition

        # Warmup calibration controller
        usage_ema_decay = 0.99,
        count_ema_decay = 0.99,
        cutoff_ema_decay = 0.99,
        calibration_threshold_lr = 0.002,

        # Elastic-stage loss-free safety controllers
        # Broad range: specialization is allowed; only near-dead / near-permanent heads move.
        usage_floor = 0.05,
        usage_ceiling = 0.95,
        head_threshold_lr = 0.005,
        global_threshold_lr = 0.002,
        controller_bound = 1.0,

        # Permanent-head training noise
        jitter_noise = 0.01,

        # nGPT self attention and FFN hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        dataset_total_steps = 65000,
        **kwargs
    ):
        # General model
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization / MLM
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # Router
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.router_grad_clip = router_grad_clip
        self.ste_temperature = ste_temperature
        self.margin_delta = margin_delta
        self.margin_lambda = margin_lambda

        self.dense_router_warmup_steps = int(dense_router_warmup * dataset_total_steps)
        self.calibration_router_warmup_steps = int(calibration_router_warmup * dataset_total_steps)
        self.elastic_start_step = self.dense_router_warmup_steps + self.calibration_router_warmup_steps
        self.elastic_transition_steps = max(1, int(elastic_transition * dataset_total_steps))

        self.usage_ema_decay = usage_ema_decay
        self.count_ema_decay = count_ema_decay
        self.cutoff_ema_decay = cutoff_ema_decay
        self.calibration_threshold_lr = calibration_threshold_lr
        self.usage_floor = usage_floor
        self.usage_ceiling = usage_ceiling
        self.head_threshold_lr = head_threshold_lr
        self.global_threshold_lr = global_threshold_lr
        self.controller_bound = controller_bound
        self.jitter_noise = jitter_noise

        # Validate the hard routing laws once in Python. These are configuration checks,
        # not tensor-dependent branches in the XLA training graph.
        elastic = num_attention_heads - num_permanent_heads
        min_elastic = head_target_min - num_permanent_heads
        max_elastic = head_target_max - num_permanent_heads
        if not (0 <= min_elastic <= max_elastic <= elastic):
            raise ValueError(
                f"Invalid Phase13 head laws: permanent={num_permanent_heads}, "
                f"min={head_target_min}, max={head_target_max}, total={num_attention_heads}"
            )
        if not (head_target_min <= head_target_center <= head_target_max):
            raise ValueError("head_target_center must lie between min and max")
        if ste_temperature <= 0:
            raise ValueError("ste_temperature must be > 0")
        center_elastic = head_target_center - num_permanent_heads
        if not (0 < center_elastic < elastic):
            raise ValueError("Phase13A-v2 fixed-center warmup requires 0 < center_elastic < elastic")
        if not (0.0 <= usage_floor < usage_ceiling <= 1.0):
            raise ValueError("usage_floor/usage_ceiling must satisfy 0 <= floor < ceiling <= 1")

        # nGPT
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.dataset_total_steps = dataset_total_steps

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# NOVEL: Multi-Latent Summary Router to decide which heads to use
# Phase 13A: elastic threshold router.
#
# Design split:
#   * q_down_proj + latent pooling: learn WHAT the sequence contains.
#   * cosine head scores: learn WHICH heads are useful.
#   * hard min/max laws: guarantee legal cardinality without a loss.
#   * global_threshold buffer: controls long-run compute budget.
#   * head_thresholds buffer: rescues dead heads / restrains monopolies.
#   * margin loss: makes ON/OFF decisions confident.
#
# The controller buffers are NOT Parameters, so Adam never updates them.
# The trainer updates them once per optimizer step after cross-replica averaging.
class HELMMultiViewRouter(nn.Module):
    """Sequence-level elastic head router with a three-stage training curriculum.

    Stage A (dense): all elastic heads execute. The router still receives CE gradient
    through the STE surrogate, but there is no margin loss and no controller mutation.

    Stage B (calibration): exactly the center number of elastic heads execute via a
    fixed-shape TopK. A loss-free per-head threshold controller equalizes *training
    opportunity during warmup only*, while the layer cutoff EMA tracks the score
    boundary between the last selected and first unselected head.

    Stage C (elastic): independent threshold routing becomes the actual forward rule.
    The hard min/max laws remain as safety constraints. Margin and loss-free safety
    controllers ramp in smoothly after the switch.

    All training masks remain [B,E]. There are no data-dependent tensor shapes.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads
        self.min_elastic_heads = config.head_target_min - config.num_permanent_heads
        self.center_elastic_heads = config.head_target_center - config.num_permanent_heads
        self.max_elastic_heads = config.head_target_max - config.num_permanent_heads

        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias=config.bias,
        )
        self.l_i_weights = nn.Parameter(torch.ones(config.num_router_latents))

        # Learned semantic routing directions. Magnitude is removed in forward() so
        # Adam can only improve direction/alignment, not cheat by inflating logit norm.
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            self.num_elastic_candidates,
            bias=False,
        )

        # ---------------- NON-ADAM CONTROLLER STATE ----------------
        self.register_buffer(
            "head_thresholds",
            torch.zeros(self.num_elastic_candidates, dtype=torch.float32),
            persistent=True,
        )
        self.register_buffer(
            "global_threshold",
            torch.tensor(0.0, dtype=torch.float32),
            persistent=True,
        )

        center_fraction = self.center_elastic_heads / float(self.num_elastic_candidates)
        self.register_buffer(
            "usage_ema",
            torch.full(
                (self.num_elastic_candidates,),
                float(center_fraction),
                dtype=torch.float32,
            ),
            persistent=True,
        )
        self.register_buffer(
            "count_ema",
            torch.tensor(float(config.head_target_center), dtype=torch.float32),
            persistent=True,
        )

        # Last-forward telemetry / controller observations (not checkpoint state).
        self.save_cosine_scores = None
        self.save_adjusted_scores = None
        self.save_router_margin = None
        self.save_surrogate_scores = None
        self.save_base_mask = None
        self.save_elastic_mask = None
        self.save_hard_mask = None
        self.save_total_head_count = None
        self.save_base_total_head_count = None
        self.save_min_law_trigger = None
        self.save_max_law_trigger = None
        self.save_shadow_min_law_trigger = None
        self.save_center_cutoff = None
        self.save_routing_stage = None
        self.save_margin_ramp = None
        self.save_margin_loss = None

    def _fixed_cardinality_mask(self, scores, k):
        """Fixed-shape [B,E] mask containing exactly top-k scores."""
        top_indices = torch.topk(scores, k=k, dim=-1, largest=True, sorted=False).indices
        return torch.zeros_like(scores).scatter(-1, top_indices, 1.0)

    def _center_mask_and_cutoff(self, scores):
        """Return fixed-center mask and midpoint cutoff between rank k and k+1.

        The midpoint is an observed threshold that would reproduce the center fanout
        for this example. During Stage B the trainer tracks its EMA, giving Stage C a
        calibrated population-level starting threshold rather than assuming zero.
        """
        k = self.center_elastic_heads
        vals, inds = torch.topk(scores, k=k + 1, dim=-1, largest=True, sorted=True)
        selected = inds[:, :k]
        center_mask = torch.zeros_like(scores).scatter(-1, selected, 1.0)
        cutoff = 0.5 * (vals[:, k - 1] + vals[:, k])
        return center_mask, cutoff

    def forward(self, hidden_states, step_tensor):
        # ----- Multi-view sequence summary -----
        q_down = justnorm(self.q_down_proj.weight, dim=1).to(hidden_states.dtype)
        scanner = F.linear(hidden_states, q_down)                         # [B,S,R]
        scanner_weights = F.softmax(self.scale * scanner, dim=1)         # [B,S,R]
        latents = torch.bmm(scanner_weights.transpose(1, 2), hidden_states)  # [B,R,D]

        latent_weights = F.softmax(self.l_i_weights, dim=0)
        pooled = (latents * latent_weights.view(1, -1, 1)).sum(dim=1)    # [B,D]

        # ----- Scale-fixed semantic utility -----
        pooled_unit = justnorm(pooled, dim=-1)
        head_directions = justnorm(self.q_up_proj.weight, dim=1).to(pooled_unit.dtype)
        cosine_scores = F.linear(pooled_unit, head_directions)           # [B,E] in [-1,1]

        head_thresholds = self.head_thresholds.to(cosine_scores.dtype).view(1, -1)
        global_threshold = self.global_threshold.to(cosine_scores.dtype)
        adjusted_scores = cosine_scores - head_thresholds
        router_margin = adjusted_scores - global_threshold

        # ----- Shadow elastic threshold rule (computed in every stage) -----
        base_mask = (router_margin > 0).to(cosine_scores.dtype)
        base_count = base_mask.sum(dim=-1, keepdim=True)

        min_mask = self._fixed_cardinality_mask(adjusted_scores, self.min_elastic_heads)
        need_min = base_count < float(self.min_elastic_heads)
        elastic_mask = torch.where(need_min, min_mask, base_mask)

        after_min_count = elastic_mask.sum(dim=-1, keepdim=True)
        max_mask = self._fixed_cardinality_mask(adjusted_scores, self.max_elastic_heads)
        need_max = after_min_count > float(self.max_elastic_heads)
        elastic_mask = torch.where(need_max, max_mask, elastic_mask)

        # ----- Fixed-center warmup routing + cutoff observation -----
        center_mask, center_cutoff = self._center_mask_and_cutoff(adjusted_scores)
        dense_mask = torch.ones_like(cosine_scores)

        # XLA-safe stage selection: scalar tensor predicates broadcast over [B,E].
        # No Python branch depends on tensor data and no tensor shape changes by stage.
        step_f = step_tensor.to(device=cosine_scores.device, dtype=torch.float32)
        dense_end = float(self.config.dense_router_warmup_steps)
        elastic_start = float(self.config.elastic_start_step)
        is_dense = step_f < dense_end
        is_calibration = (step_f >= dense_end) & (step_f < elastic_start)
        is_elastic = step_f >= elastic_start

        hard_mask = torch.where(
            is_dense,
            dense_mask,
            torch.where(is_calibration, center_mask, elastic_mask),
        )

        # ----- Backward surrogate -----
        # Sigmoid is ONLY a derivative surrogate. It never determines cardinality.
        tau = float(self.config.ste_temperature)
        surrogate_scores = torch.sigmoid(router_margin / tau)
        ste_mask = hard_mask.detach() - surrogate_scores.detach() + surrogate_scores

        hard_elastic_count = hard_mask.sum(dim=-1)
        total_head_count = hard_elastic_count + float(self.config.num_permanent_heads)
        base_total_head_count = base_count.squeeze(-1) + float(self.config.num_permanent_heads)

        # ----- Margin loss: OFF in A/B, smoothly ramped after elastic switch -----
        transition = float(max(1, self.config.elastic_transition_steps))
        margin_ramp = torch.clamp((step_f - elastic_start) / transition, 0.0, 1.0)

        if self.training:
            delta = float(self.config.margin_delta)
            hard_detached = hard_mask.detach()
            usage = self.usage_ema.detach().to(router_margin.dtype).view(1, -1)

            # Symmetric safety weighting: do not let the margin loss fight the
            # controller by pushing a nearly-dead head further OFF or a nearly-
            # permanent head further ON.
            floor = max(float(self.config.usage_floor), 1e-6)
            ceiling = min(float(self.config.usage_ceiling), 1.0 - 1e-6)
            off_health_weight = torch.clamp(usage / floor, 0.0, 1.0)
            on_health_weight = torch.clamp((1.0 - usage) / (1.0 - ceiling), 0.0, 1.0)

            on_penalty = torch.relu(delta - router_margin).square()
            off_penalty = torch.relu(delta + router_margin).square()
            raw_margin_loss = (
                hard_detached * on_health_weight * on_penalty
                + (1.0 - hard_detached) * off_health_weight * off_penalty
            ).mean()
            self.margin_loss = (
                float(self.config.margin_lambda) * margin_ramp * raw_margin_loss
            )
        else:
            self.margin_loss = cosine_scores.new_zeros(())

        # ----- Telemetry / controller observations -----
        self.save_cosine_scores = cosine_scores.detach()
        self.save_adjusted_scores = adjusted_scores.detach()
        self.save_router_margin = router_margin.detach()
        self.save_surrogate_scores = surrogate_scores.detach()
        self.save_base_mask = base_mask.detach()
        self.save_elastic_mask = elastic_mask.detach()
        self.save_hard_mask = hard_mask.detach()
        self.save_total_head_count = total_head_count.detach()
        self.save_base_total_head_count = base_total_head_count.detach()
        self.save_shadow_min_law_trigger = need_min.squeeze(-1).to(torch.float32).detach()
        self.save_min_law_trigger = (
            need_min.squeeze(-1).to(torch.float32) * is_elastic.to(torch.float32)
        ).detach()
        self.save_max_law_trigger = (
            need_max.squeeze(-1).to(torch.float32) * is_elastic.to(torch.float32)
        ).detach()
        self.save_center_cutoff = center_cutoff.detach()
        self.save_routing_stage = (
            is_calibration.to(torch.float32) + 2.0 * is_elastic.to(torch.float32)
        ).detach()
        self.save_margin_ramp = margin_ramp.detach()
        self.save_margin_loss = self.margin_loss.detach()

        # Add permanent heads. Training attention remains dense/static-shape.
        router_mask = ste_mask.view(ste_mask.size(0), -1, 1, 1)
        if self.config.num_permanent_heads > 0:  # compile-time config branch only
            permanent = torch.ones(
                ste_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype,
            )
            router_mask = torch.cat((permanent, router_mask), dim=1)

        return router_mask.to(hidden_states.dtype)

    @torch.no_grad()
    def get_observation_means(self):
        """Fixed-size observations from the last forward."""
        usage_mean = self.save_hard_mask.float().mean(dim=0)             # [E]
        count_mean = self.save_total_head_count.float().mean()           # scalar
        cutoff_mean = self.save_center_cutoff.float().mean()             # scalar
        return usage_mean, count_mean, cutoff_mean

    @torch.no_grad()
    def update_controller(self, usage_mean, count_mean, cutoff_mean, global_step):
        """Loss-free controller update, called once per optimizer step.

        This method runs outside forward() after cross-replica averaging. The only
        Python stage branch is on the host integer global_step, so it does not create
        data-dependent branches inside the XLA model graph.
        """
        usage_mean = usage_mean.to(device=self.usage_ema.device, dtype=torch.float32)
        count_mean = count_mean.to(device=self.count_ema.device, dtype=torch.float32)
        cutoff_mean = cutoff_mean.to(device=self.global_threshold.device, dtype=torch.float32)

        dense_end = int(self.config.dense_router_warmup_steps)
        elastic_start = int(self.config.elastic_start_step)

        # Stage A: no controller mutation. The router gets CE/STE gradients while
        # every head is executed, avoiding early lock-in from arbitrary hard choices.
        if global_step < dense_end:
            return

        du = float(self.config.usage_ema_decay)
        dc = float(self.config.count_ema_decay)
        self.usage_ema.mul_(du).add_(usage_mean, alpha=(1.0 - du))
        self.count_ema.mul_(dc).add_(count_mean, alpha=(1.0 - dc))

        # Stage B: fixed-center forward routing. Calibrate each threshold so no fixed
        # subset can monopolize warmup, and track the observed center cutoff EMA.
        if global_step < elastic_start:
            target_rate = self.center_elastic_heads / float(self.num_elastic_candidates)
            usage_error = self.usage_ema - float(target_rate)
            self.head_thresholds.add_(
                float(self.config.calibration_threshold_lr) * usage_error
            )

            beta = float(self.config.cutoff_ema_decay)
            self.global_threshold.mul_(beta).add_(cutoff_mean, alpha=(1.0 - beta))

        # Stage C: only broad safety regulation remains. Crucially, use RAW distance
        # from the safety band rather than dividing by floor/ceiling width. The old
        # normalized controller moved a fully dead head by the maximum update every
        # step and could hit +/-1 in ~500 steps.
        else:
            transition = float(max(1, self.config.elastic_transition_steps))
            ramp = min(1.0, max(0.0, (float(global_step) - float(elastic_start)) / transition))

            floor = float(self.config.usage_floor)
            ceiling = float(self.config.usage_ceiling)
            low_deficit = torch.relu(floor - self.usage_ema)
            high_excess = torch.relu(self.usage_ema - ceiling)
            self.head_thresholds.add_(
                ramp * float(self.config.head_threshold_lr) * (high_excess - low_deficit)
            )

            budget_error = (
                self.count_ema - float(self.config.head_target_center)
            ) / float(max(1, self.num_elastic_candidates))
            self.global_threshold.add_(
                ramp * float(self.config.global_threshold_lr) * budget_error
            )

        bound = float(self.config.controller_bound)
        self.head_thresholds.clamp_(-bound, bound)
        self.global_threshold.clamp_(-bound, bound)


class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.d_head = config.d_head if config.d_head is not None else (config.hidden_size // config.num_attention_heads)
        self.total_head_dim = self.num_attention_heads * self.d_head   
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.total_head_dim))  # was: self.hidden_size

        # Output Matrix
        self.output = nn.Linear(
            self.total_head_dim,      # was: config.hidden_size
            config.hidden_size,
            bias = config.bias
        )

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        # Obtain Hidden Size
        batch_size, seq_len, _ = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)


        eval_backend = self._eval_backend

        # Reshape q,k,v
        # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

        # Reshape q,k,v
        # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)


        # TRAINING / TPU MODE
        if (self.training or eval_backend == "dense"):

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Apply Attention
            # Scale by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head),
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # FLEX ATTENTION (for GPUs)
        elif eval_backend == "flex" and batch_size > 1:

             # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Router_mask scores, 1 or 0 or sigmoid scaling
            # [b, num_attention_heads, 1 , 1] -> [batch, num_attention_heads]
            active = (router_mask[:, :, 0, 0] > 0)

            # Boolean attention mask
            # [batch_size, 1, 1, seq_len] -> [batch, seq_len]
            key_valid = (attention_mask[:, 0, 0, :] >=0)

            # mask_mod: attend / calculate only if the head is on and its not a padding token
            def mask_mod(bi, hi, qi, ki):
                return active[bi, hi] & key_valid[bi, ki]

            # Prep the block to be passed into flex attention
            block_mask = self._build_block_mask(
                mask_mod, batch_size, self.num_attention_heads,seq_len, q.device
            )

            # Apply flex attention
            context_layer = self._flex_attn(
                q, k, v, block_mask = block_mask, scale = math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Apply router mask to 0 the heads of the context layer
            # [batch, num attention heads, seq_len, head dim] (router_mask [batch, num_attention_heads, 1,1] was broadcasted)
            context_layer = context_layer * router_mask.expand_as(context_layer)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # Single query effieincy
        else:

            # This path only looks at batch element 0's router decisions (see below), so
            # it is only correct for batch_size == 1 -- each example's active heads are
            # data-dependent, so silently reusing example 0's mask for other examples
            # would produce wrong outputs for them instead of a loud failure.
            assert batch_size == 1, (
                f"HELMSelfAttention's 'gather' eval backend only supports batch_size == 1 "
                f"(got batch_size={batch_size}); use backend='flex' for batched inference."
            )

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # q, k, v are already [b, num_attention_heads, seq_len, d_head] from the
            # shared reshape/permute above -- no need to reshape them again here.

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor to q and k
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced

            # Flash Attention (only for GPUs where on-the-fly splicing can exist)
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims].to(context_reshaped.dtype)
            sliced_bias = None if self.output.bias is None else self.output.bias.to(context_reshaped.dtype)

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=sliced_bias)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention (which defines RotaryEmbeddigs) + HELMMLP
# This is 1 transformer layer
class HELMBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask, step_tensor):
        router_mask = self.mlt_vw_rtr(hidden_states, step_tensor)
        margin_loss = self.mlt_vw_rtr.margin_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, margin_loss


class HELMModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList([HELMBlock(config) for _ in range(config.num_hidden_layers)])

    def forward(self, input_ids, attention_mask, current_step=None):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        # current_step is runtime DATA, not a Python routing branch. All three masks
        # are built every forward and torch.where selects the active stage at fixed shape.
        if current_step is None:
            step_tensor = torch.tensor(float("inf"), device=input_ids.device, dtype=torch.float32)
        elif isinstance(current_step, torch.Tensor):
            step_tensor = current_step.to(device=input_ids.device, dtype=torch.float32)
        else:
            step_tensor = torch.tensor(float(current_step), device=input_ids.device, dtype=torch.float32)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)
        total_margin_loss = hidden_states.new_zeros(())

        for block in self.blocks:
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, margin_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    step_tensor,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states, margin_loss = block(hidden_states, attention_mask, step_tensor)
            total_margin_loss = total_margin_loss + margin_loss

        return hidden_states, total_margin_loss


class HELMForMaskedLM(PreTrainedModel):

    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.vocab_size, bias=config.bias)
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # q_up_proj is included now because its stored direction is the router's
        # semantic head direction. Forward still normalizes it functionally as well.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight",
            "mlt_vw_rtr.q_up_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_router_observation_means(self):
        """Return fixed-size [L,E] usage plus [L] count/cutoff observations."""
        usage = []
        counts = []
        cutoffs = []
        for block in self.model.blocks:
            u, c, q = block.mlt_vw_rtr.get_observation_means()
            usage.append(u)
            counts.append(c)
            cutoffs.append(q)
        return (
            torch.stack(usage, dim=0),
            torch.stack(counts, dim=0),
            torch.stack(cutoffs, dim=0),
        )

    @torch.no_grad()
    def update_router_controllers(self, usage_mean, count_mean, cutoff_mean, global_step):
        """Apply globally synchronized observations once per optimizer step."""
        for i, block in enumerate(self.model.blocks):
            block.mlt_vw_rtr.update_controller(
                usage_mean[i], count_mean[i], cutoff_mean[i], global_step
            )

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}

        for i, block in enumerate(self.model.blocks):
            router = block.mlt_vw_rtr

            # Router decision geometry
            telemetry[f"layer_{i}_cosine_scores"] = router.save_cosine_scores.float().cpu()
            telemetry[f"layer_{i}_adjusted_scores"] = router.save_adjusted_scores.float().cpu()
            telemetry[f"layer_{i}_router_margin"] = router.save_router_margin.float().cpu()
            telemetry[f"layer_{i}_surrogate_scores"] = router.save_surrogate_scores.float().cpu()
            telemetry[f"layer_{i}_base_mask"] = router.save_base_mask.float().cpu()
            telemetry[f"layer_{i}_elastic_mask"] = router.save_elastic_mask.float().cpu()
            telemetry[f"layer_{i}_hard_mask"] = router.save_hard_mask.float().cpu()
            telemetry[f"layer_{i}_elastic_head_ratio"] = router.save_hard_mask.float().mean().item()
            telemetry[f"layer_{i}_routing_stage"] = router.save_routing_stage.float().cpu().item()
            telemetry[f"layer_{i}_margin_ramp"] = router.save_margin_ramp.float().cpu().item()
            telemetry[f"layer_{i}_center_cutoff_mean"] = router.save_center_cutoff.float().mean().cpu().item()
            telemetry[f"layer_{i}_shadow_min_law_trigger_rate"] = router.save_shadow_min_law_trigger.float().mean().item()
            telemetry[f"layer_{i}_total_head_count_mean"] = router.save_total_head_count.float().mean().item()
            telemetry[f"layer_{i}_base_total_head_count_mean"] = router.save_base_total_head_count.float().mean().item()
            telemetry[f"layer_{i}_min_law_trigger_rate"] = router.save_min_law_trigger.float().mean().item()
            telemetry[f"layer_{i}_max_law_trigger_rate"] = router.save_max_law_trigger.float().mean().item()
            telemetry[f"layer_{i}_margin_loss"] = router.save_margin_loss.float().item()

            # Non-Adam controller state
            thresholds = router.head_thresholds.detach().float().cpu()
            usage = router.usage_ema.detach().float().cpu()
            telemetry[f"layer_{i}_global_threshold"] = router.global_threshold.detach().float().cpu().item()
            telemetry[f"layer_{i}_head_thresholds_hist"] = thresholds
            telemetry[f"layer_{i}_head_thresholds_mean"] = thresholds.mean().item()
            telemetry[f"layer_{i}_head_thresholds_std"] = thresholds.std(unbiased=False).item()
            telemetry[f"layer_{i}_head_thresholds_min"] = thresholds.min().item()
            telemetry[f"layer_{i}_head_thresholds_max"] = thresholds.max().item()
            telemetry[f"layer_{i}_usage_ema_hist"] = usage
            telemetry[f"layer_{i}_usage_ema_mean"] = usage.mean().item()
            telemetry[f"layer_{i}_usage_ema_min"] = usage.min().item()
            telemetry[f"layer_{i}_usage_ema_max"] = usage.max().item()
            telemetry[f"layer_{i}_count_ema"] = router.count_ema.detach().float().cpu().item()
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().float().cpu()

            # Self-attention nGPT telemetry
            sqk_tensor = block.attn.sqk.detach().cpu()
            telemetry[f"layer_{i}_sqk_mean"] = sqk_tensor.mean().item()
            telemetry[f"layer_{i}_sqk_std"] = sqk_tensor.std().item()
            telemetry[f"layer_{i}_sqk_hist"] = sqk_tensor

            # MLP nGPT telemetry
            mlp = block.mlp
            attn_alpha_tensor = mlp.attn_alpha.detach().cpu()
            telemetry[f"layer_{i}_attn_alpha_mean"] = attn_alpha_tensor.mean().item()
            telemetry[f"layer_{i}_attn_alpha_std"] = attn_alpha_tensor.std().item()
            telemetry[f"layer_{i}_attn_alpha_hist"] = attn_alpha_tensor

            mlp_alpha_tensor = mlp.mlp_alpha.detach().cpu()
            telemetry[f"layer_{i}_mlp_alpha_mean"] = mlp_alpha_tensor.mean().item()
            telemetry[f"layer_{i}_mlp_alpha_std"] = mlp_alpha_tensor.std().item()
            telemetry[f"layer_{i}_mlp_alpha_hist"] = mlp_alpha_tensor

            suv_tensor = mlp.suv.detach().cpu()
            telemetry[f"layer_{i}_suv_mean"] = suv_tensor.mean().item()
            telemetry[f"layer_{i}_suv_std"] = suv_tensor.std().item()
            telemetry[f"layer_{i}_suv_hist"] = suv_tensor

        sz_tensor = self.sz.detach().cpu()
        telemetry["lm_head_sz_mean"] = sz_tensor.mean().item()
        telemetry["lm_head_sz_std"] = sz_tensor.std().item()
        telemetry["lm_head_sz_hist"] = sz_tensor
        return telemetry

    def forward(self, input_ids, attention_mask, current_step=None, easiness_score=None):
        # easiness_score remains a compatibility argument only and is never consumed.
        # current_step controls the fixed-shape routing curriculum.
        features, total_margin_loss = self.model(
            input_ids, attention_mask, current_step=current_step
        )
        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)
        unscaled_logits = cast_linear(features, self.classifier)
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits
        return logits, total_margin_loss


Writing model.py


In [12]:
%%writefile parallel_hardware_trainer.py
DOES_RESUME_FROM_WORK = False
TESTING_MODE = False

# Imports (Manifesting that my loss curve will look like this)
import io
import os
import re
import sys
import time
import json
import site
import math
import glob
import torch
import wandb
import warnings
import importlib
import numpy as np
import multiprocessing
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset
from torch.optim.lr_scheduler import LambdaLR
from typing import Optional, List, Union, ClassVar, Dict, Any
from huggingface_hub import hf_hub_download, create_repo, HfApi
from huggingface_hub.utils import RepositoryNotFoundError, EntryNotFoundError


# # Disable Progress Bars
# try:
#     from huggingface_hub.utils import disable_progress_bars
#     disable_progress_bars()
# except ImportError:
#     pass

# Ensure PJRT runtime gets selected, not XRT
for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
    os.environ.pop(key, None)
os.environ["PJRT_DEVICE"] = "TPU"
# Add the framework quarantine just in case!
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Prevent C++ thread deadlocks during 10B token streaming
os.environ["OMP_NUM_THREADS"] = "1"
import pyarrow as pa
pa.set_cpu_count(1)
pa.set_io_thread_count(1)

# Tell everything to stay away from the TPU except PyTorch
os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"

# Gets modified to tell child processes to return cleanly
SHUTDOWN_FILE = "/tmp/SHUTDOWN_REQUESTED"
# This lets the restarting launching script whether user intentionally ended program or not
USER_STOP_MARKER = "/tmp/USER_STOPPED_TRAINING"

# Force Path Refresh
if 'site' in sys.modules:
    importlib.reload(site)

# Get HF_TOKEN and WANDB_API_KEY
def get_secret(key_name):

    # Try Colab
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except:
        pass

    # Try Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except:
        pass

    # Local Env
    return os.getenv(key_name)

# Get the float value of something (mainly for losses)
def to_float(x):
    return x.item() if hasattr(x, 'item') else float(x)


# ==================================================
# HardwareConfig
# Keeps track of which device to use  what device-dependent values to use
# Not Static (Changes State)
# ==================================================

@dataclass
class HardwareConfig:

    HARDWARE_PROFILES = {
        "v5e-8": {
            "ws": 8, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 2, "use_ckpt": False, "sl": 1024},
            1: {"mb": 2, "use_ckpt": False, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "v5e-1": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 3, "use_ckpt": False, "sl": 1024},
            1: {"mb": 2, "use_ckpt": False, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "v6e-1": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 16, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": False, "sl": 2048},
            2: {"mb": 2, "use_ckpt": False, "sl": 4096},
        },
        "t4*2": {
            "ws": 2, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 4, "use_ckpt": False, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "t4": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 4, "use_ckpt": False, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "g4": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 32, "use_ckpt": False, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "l4": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 8, "use_ckpt": False, "sl": 1024},
            1: {"mb": 2, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "p100": {
            "ws": 1, "target": 128, "dtype": torch.float16, "use_scaler": True,
            0: {"mb": 8, "use_ckpt": True, "sl": 1024},
            1: {"mb": 1, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "a100": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 32, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "h100": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 32, "use_ckpt": False, "sl": 1024},
            1: {"mb": 4, "use_ckpt": True, "sl": 2048},
            2: {"mb": 1, "use_ckpt": True, "sl": 4096},
        },
        "cpu": {
            "ws": 1, "target": 128, "dtype": torch.bfloat16,"use_scaler": False,
            0: {"mb": 1, "use_ckpt": False, "sl": 1024},
            1: {"mb": 1, "use_ckpt": False, "sl": 2048},
            2: {"mb": 1, "use_ckpt": False, "sl": 4096},
        }
    }

    hardware_string: str = "v5e-8 tpu"
    # hardware_string: str = "t4*2 gpu"
    hf_token: str = ""


    # These will be overwritten when we step thru the curriculum, but just place_holders for now
    world_size: int = 1
    target_gbs: int = 128
    dtype: torch.dtype = torch.float16
    use_scaler: bool = True
    batch_size: int = 16
    grad_accum_steps: int = 1
    hardware_profile: Dict[Union[str, int], Any] = field(
        default_factory=lambda: HardwareConfig.HARDWARE_PROFILES["cpu"]
    )
    device_type: str = "cpu"
    validation_step_num: int = 50 # or until exhaustion


# ==================================================
# MLMDataConfig
# Sets dataset repo, curriculum levels, and MLM parameters
# Static (Remains the same)
# ==================================================

@dataclass
class MLMDataConfig:
    data_repo_id: str = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
    curriculum: bool = True
    curriculum_subset_names: List[str] = field(
        default_factory=lambda: ["seq_1024", "seq_2048", "seq_4096"]
    )
    curriculum_parquet_start_index: List[int] = field(
        default_factory=lambda: [0, 72, 92]
    )

    train_split: str = "train"
    validation_split: str = "validation"
    tokenizer_name: str = "answerdotai/ModernBERT-base"
    mlm_probability: float = 0.3
    mlm_use_span_masking: bool = True
    mlm_span_length: int = 3
    # Format String without the f
    glob_pattern: str = "data/{subset_name}/{split}-*.parquet"
    # Keep easiness in the collator ONLY for post-hoc router telemetry.
    # Phase13A never passes this value into the model.
    use_easiness: bool = True
    # Stopping index so the model will stop training when it hits this index
    parquet_stop_index: int = 10


# ==================================================
# CheckpointConfig
# Sets Checkpoint related fields
# Static (Remains the same)
# ==================================================

@dataclass
class CheckpointConfig:
    model_repo_id: str = "JamesResearch1216/phase7a-warmup-v2-32h-64d"
    wandb_entity: str = "jhui16-university-of-maryland"
    wandb_project: str = "HELM-v1-10B-Run"
    wandb_name: str = "phase7a-warmup-v2"
    hf_token: str = ""
    wandb_key: str = ""
    use_wandb: bool = True

    # How to use interval_dict:
    #  - Let k_i be the ith key in interval_dict
    #  - Let v_i be the ith value in interval_dict
    #  - For step k_i to k_i+1, save a checkpoint every v_i steps
    #  - After the last k_i, save every v_i for the rest of the duration
    interval_dict: Dict[int, int] = field(
        default_factory=lambda: {0: 100, 1000: 200, 5000: 500}
    )
    # True: let step 0 = latest_step
    # False: let step 0 = 0
    start_from_global: bool = True



class MLMDataStrategy:

    # Initialize (Different for each hardware)
    def __init__(self, rank = 0, world_size = 1, is_tpu = False, config: Optional[MLMDataConfig] = None, hf_token=None):
        self.rank = rank
        self.world_size = world_size
        self.is_tpu = is_tpu
        self.config = config
        self.hf_token = hf_token

    # Input:
    # - index number i
    # - delete_prev_parquet_request flag
    # Output:
    # - local file_path name for dataset shard
    # - # of total rows in the shard
    # Load the ith parquet into runtime
    def download_parquet(self, is_train: bool, index = 0):

        # Finding Correct curriculum
        curriculum_level = 0
        for level in range(0, len(self.config.curriculum_parquet_start_index)):
            if (index >= self.config.curriculum_parquet_start_index[level]):
                curriculum_level = level

        # Set up dataset types
        dataset_type = "train" if is_train else "validation"

        # Fix indices for validation due to my weird validation parquet naming
        # the plus 1 because 1 indexed
        index = index if is_train else curriculum_level

        # Get parquet file path
        parquet_file_path = f"data/{self.config.curriculum_subset_names[curriculum_level]}/{dataset_type}-{index:05d}.parquet"

        local_storage_dir = "./local_parquet_shards"
        os.makedirs(local_storage_dir, exist_ok=True)

        # Form file path that potientially is already in file_path
        local_path = os.path.join(local_storage_dir, parquet_file_path)
        if os.path.exists(local_path):
            try:
                parquet_metadata = pq.read_metadata(local_path)
                num_rows = parquet_metadata.num_rows
                return local_path, num_rows, curriculum_level
            except Exception as e:
                if self.rank == 0:
                    print(f"Error when trying to access {local_path}: {e}. Redownloading instead")

        # Download
        try:
            parquet_file_path = hf_hub_download(
                repo_id = self.config.data_repo_id,
                filename = parquet_file_path,
                repo_type = "dataset",
                token = self.hf_token,
                local_dir = local_storage_dir,
                local_dir_use_symlinks = False
            )

            # Get # of rows
            parquet_metadata = pq.read_metadata(local_path)
            num_rows = parquet_metadata.num_rows

            return parquet_file_path, num_rows, curriculum_level

        except Exception as e:
            print(f"Failed to download {parquet_file_path}: {e}")
            return "", 0, 0

    # Delete Parquet
    def delete_parquet(self, parquet_file_path: str):
        try:
            if parquet_file_path and os.path.exists(parquet_file_path):
                os.remove(parquet_file_path)
            else:
                print(f"File not found for deletion: {parquet_file_path}")
                return -1
        except Exception as e:
            print(f"Failed to delete {parquet_file_path}: {e}")
            return -1



    # Create get_mlm_data_loader function
    # Note: This only bascially works with the dataset created by prepare_data.py
    def get_mlm_data_loader(
        self,
        parquet_file_path: str,
        collate_fn = None,
        skip_rows = 0,
        batch_size = 1,
        parquet_index = 1,
        is_train = True,
        ):

        # Get HF Dataset obj from parquet
        dataset = Dataset.from_parquet(path_or_paths = parquet_file_path, keep_in_memory = True)

        # Shuffle after you shard
        # Make sure you set a seed to ensure the I don't use the same data again
        # + index to keep this more random but predictable for reproductibility
        dataset = dataset.shuffle(seed = 67 + parquet_index)

        # Skip examples after you shuffle based on the specific seed:
        if skip_rows > 0 and is_train:

            # If skip rows is somehow over the dataset, return an empty dataset
            if skip_rows >= len(dataset):
                dataset = dataset.select(range(0))   # empty
            # Else skip skip_rows and grab remaining data
            else:
                dataset = dataset.select(range(skip_rows, len(dataset)))

        # If we're Parallel Processing, Shard the Data so that each device gets a different slice of data
        if self.world_size > 1 and is_train:
            dataset = dataset.shard(num_shards = self.world_size, index = self.rank)

        # Dataloader
        data_loader = DataLoader(
            dataset,
            batch_size = batch_size,
            num_workers = 0,
            drop_last = True,
            pin_memory = False,
            collate_fn = collate_fn
        )

        # Return data_loader
        return data_loader



class HardwareDriver:

    # Initialize Hardware
    def __init__(self, hw_config: HardwareConfig, data_config: MLMDataConfig, ckpt_config: CheckpointConfig):
        self.hw_config = hw_config
        self.data_config = data_config
        self.ckpt_config = ckpt_config
        # Call _parse_hardware here
        self.hw_config.hardware_profile = self._parse_hardware()


    # Parse hardware based on the curriculum level
    def _parse_hardware(self):
        # get hardware_string (with formatting)
        hardware_string = self.hw_config.hardware_string.lower().replace(" ", "")

        # Ensure that the hardware_string is valid
        profile = None
        for key in self.hw_config.HARDWARE_PROFILES:
            if (key in hardware_string):
                profile = self.hw_config.HARDWARE_PROFILES[key]
                break

        # Default to cpu if none were matching
        if not profile:
            profile = self.hw_config.HARDWARE_PROFILES["cpu"]
            warnings.warn("⚠️ hardware_string did not match any in HARDWARE_PROFILES. Using \"cpu\"", UserWarning)

        # Add attributes to config common to all curriculum levels
        #   - world_size
        #   - targt_gbs (global batch size)
        #   - dtype (data type)
        #   - use_scaler
        self.hw_config.world_size = profile["ws"]
        self.hw_config.target_gbs = profile["target"]
        self.hw_config.dtype = profile["dtype"]
        self.hw_config.use_scaler = profile["use_scaler"]

        # Get device type (save to hw_config)
        if "tpu" in hardware_string:
            self.hw_config.device_type = "tpu"
        elif any(x in hardware_string for x in ["gpu", "cuda", "a100", "p100", "h100", "t4", "l4"]):
            self.hw_config.device_type = "cuda"
        else:
            self.hw_config.device_type = "cpu"

        # Return the profile to use in train_worker
        return profile

    # Launch function: Spawn all the workers and make them run the worker_function
    def launch(self, worker_fn):

        # Define these for convience
        world_size = self.hw_config.world_size
        device = self.hw_config.device_type

        # If parallel processing
        if world_size > 1:
            if device == "tpu":
                import torch_xla.distributed.xla_multiprocessing as xmp
                xmp.spawn(worker_fn, args=(self.hw_config, self.data_config, self.ckpt_config), start_method='spawn')
            elif device == "cuda":
                import random
                import torch.multiprocessing as mp

                # Set up Multi-GPU network
                os.environ['MASTER_ADDR'] = 'localhost'
                os.environ['MASTER_PORT'] = str(random.randint(10001, 19999))


                mp.spawn(worker_fn, args=(self.hw_config, self.data_config, self.ckpt_config), nprocs=world_size)
        else:
            # Single Device Execution (Rank 0)
            worker_fn(0, self.hw_config, self.data_config, self.ckpt_config)



class CheckpointDriver:

    # Initialize Checkpoint Driver
    def __init__(self, hw_config: HardwareConfig, data_config: MLMDataConfig, ckpt_config: CheckpointConfig, rank: int, world_size: int):
        self.hw_config = hw_config
        self.data_config = data_config
        self.ckpt_config = ckpt_config
        self.rank = rank
        self.world_size = world_size
        self.api = HfApi(token=self.ckpt_config.hf_token)
        self.actual_resume_step = None
        self.total_rows_dict = None
        self.easiness_dict = None
        self.use_easiness = self.data_config.use_easiness

        # Just print to ensure shit is moving
        if rank == 0:
            print("⏳ Loading Checkpoint Driver...")

        self.training_state = self.get_training_state_from_hub()


    # Smart Barrier (Rendezvous) to prevent data races & ensure all devices make it to certain step
    def _smart_barrier(self, name="barrier"):
        if self.hw_config.world_size <= 1:
            return  # No synchronization needed for single device

        if self.hw_config.device_type == "tpu":
            import torch_xla.core.xla_model as xm
            xm.rendezvous(name)
        elif self.hw_config.device_type == "cuda":
            import torch.distributed as dist
            if dist.is_initialized():
                dist.barrier()

    # Get the number of rows
    def _get_total_rows(self):
        from huggingface_hub import HfFileSystem
        import pyarrow.parquet as pq

        # "all" = All curriculums
        #  0 = 0th level curriculum
        #  1 = 1st level curriculum
        # etc...
        total_rows_dict = {
            "all" : 0
        }

        # Initialize HfFileSystem Object
        fs = HfFileSystem(token = self.ckpt_config.hf_token)
        repo_id = self.data_config.data_repo_id

        # Get total num rows for lr_scheduler
        for level, subset_name in enumerate(self.data_config.curriculum_subset_names):
            total_rows_dict[str(level)] = 0

            try:
                pattern = self.data_config.glob_pattern.format(
                    subset_name = subset_name,
                    split = self.data_config.train_split
                )
                parquet_files = fs.glob(f"datasets/{repo_id}/{pattern}")

                for file_path in parquet_files:

                    # binary read mode to get metadata
                    with fs.open(file_path, "rb") as f:
                        # Get metadata
                        metadata = pq.read_metadata(f)
                        # Get num_rows attr.
                        num_rows = metadata.num_rows

                        total_rows_dict[str(level)] += num_rows
                        total_rows_dict["all"] += num_rows

            except Exception as e:
                if self.rank == 0:
                    print(f"💀 Error reading metadata for {subset_name}: {e}")

        if self.rank == 0:
            for k, v in total_rows_dict.items():
                label = "ALL" if k == "all" else self.data_config.curriculum_subset_names[int(k)]
                print(f"  📊 {label}: {v:,} rows")

        return total_rows_dict

    # Only use this for easiness
    def _compute_easiness_breakpoints(self, column="easiness_score", n_breakpoints=101, max_files=5):
        from huggingface_hub import HfFileSystem
        import numpy as np

        fs = HfFileSystem(token=self.ckpt_config.hf_token)
        repo_id = self.data_config.data_repo_id
        local_dir = "./local_parquet_shards"
        os.makedirs(local_dir, exist_ok=True)

        easiness_dict = None
        all_vals = []

        # For each curriculum:
        for level, subset_name in enumerate(self.data_config.curriculum_subset_names):

            try:

                # Take all the file_paths for the parquets used to calculate the break-points easiness distribution
                pattern = self.data_config.glob_pattern.format(
                    subset_name = subset_name,
                    split = self.data_config.train_split
                )

                parquet_files = sorted(fs.glob(f"datasets/{repo_id}/{pattern}"))

                parquets_sampled = parquet_files[:max_files]

                if self.rank == 0:
                    print(f"  📥 {subset_name}: sampling {len(parquets_sampled)}/{len(parquet_files)} ...")

                level_vals = []
                downloaded_paths = []

                # Go through all the curriculum's sampled parquets
                for hf_path in parquets_sampled:

                    # hf_path looks like "datasets/user/repo/data/seq_1024/train-00000.parquet"
                    # Extract the repo-relative filename for hf_hub_download
                    # Strip the "datasets/{repo_id}/" prefix
                    prefix = f"datasets/{repo_id}/"
                    filename = hf_path[len(prefix):] if hf_path.startswith(prefix) else hf_path

                    # Download parquet
                    try:
                        local_path = hf_hub_download(
                            repo_id=repo_id,
                            filename=filename,
                            repo_type="dataset",
                            token=self.ckpt_config.hf_token,
                            local_dir=local_dir,
                            local_dir_use_symlinks=False
                        )
                        downloaded_paths.append(local_path)

                        # Read ONLY the easiness column (fast, low memory)
                        col_data = pq.read_table(
                            local_path, columns=[column]
                        ).column(column).to_numpy(zero_copy_only=False)
                        level_vals.append(np.asarray(col_data, dtype=np.float64))

                    except Exception as e:
                        if self.rank == 0:
                            print(f"    ⚠️ Failed to read {filename}: {e}")

                    # Delete the parquet
                    for path in downloaded_paths:
                        try:
                            if os.path.exists(path):
                                os.remove(path)
                        except Exception:
                            pass

                    # Collect the values
                    if level_vals:
                        level_concat = np.concatenate(level_vals)
                        level_concat = level_concat[np.isfinite(level_concat)]
                        all_vals.append(level_concat)

            except Exception as e:
                if self.rank == 0:
                    print(f"  💀 Error processing {subset_name}: {e}")

        # Compute GLOBAL breakpoints (combining all subsets)
        if all_vals:

            global_concat = np.concatenate(all_vals)
            easiness_dict = self._compute_breakpoint_payload(
                global_concat, n_breakpoints, column
            )
            if self.rank == 0:
                g = easiness_dict
                print(f"  🌍 Global easiness: n={g['n']:,} median={g['median']:.3f} "
                    f"mean={g['mean']:.3f} frac>0.5={g['frac_above_0.5']:.2f}")
        else:
            if self.rank == 0:
                print("  ⚠️ No easiness data found — using logistic fallback in model")
            easiness_dict = None

        return easiness_dict

    @staticmethod
    def _compute_breakpoint_payload(values, n_breakpoints, column):
        """Helper: given a numpy array of easiness values, return the breakpoint dict."""
        import numpy as np
        breakpoints = np.quantile(values, np.linspace(0.0, 1.0, n_breakpoints)).tolist()
        return {
            "breakpoints": breakpoints,
            "median": float(np.median(values)),
            "mean": float(np.mean(values)),
            "frac_above_0.5": float(np.mean(values > 0.5)),
            "n": int(values.size),
            "column": column,
        }



    # Initialize new checkpoint dictionary
    def _init_new_training_state(self):

        if self.rank == 0:
            print("🔢 Computing total rows per curriculum level...")
        self.total_rows_dict = self._get_total_rows()

        training_state_dict = {
            "checkpoints": {},
            "session": 0,
            "total_rows_dict" : self.total_rows_dict,
        }

        # Phase13A deliberately does not compute easiness breakpoints.
        # The raw easiness column is retained in batches for telemetry only.

        formatted_json_str = json.dumps(training_state_dict, indent = 4)
        json_bytes = formatted_json_str.encode('utf-8')
        fileobj = io.BytesIO(json_bytes)
        try:
            self.api.upload_file(
                path_or_fileobj = fileobj,
                path_in_repo = "training_state.json",
                repo_id = self.ckpt_config.model_repo_id,
                repo_type = "model",
                token = self.ckpt_config.hf_token
            )
        except Exception as e:
            print(f"Failed to push Init State {e}")

        return training_state_dict


    # Ensure that if checkpoints are deleted but appear
    def _deletion_status_updates(self, training_state):

        # Try to take the repo file's file paths
        repo_files = None
        try:
            repo_files = list(self.api.list_repo_files(repo_id = self.ckpt_config.model_repo_id))
        except RepositoryNotFoundError:
            raise RepositoryNotFoundError(f"❌ Repo: \"{self.ckpt_config.model_repo_id}\" was not found when trying to update deletion status")


        # Loop Through every checkpoint and switch the status if necessary
        for ckpt_vals in training_state["checkpoints"].values():
            if ckpt_vals["file"] == "" or not ckpt_vals["file"] in repo_files:
                ckpt_vals["status"] = "deleted"
                ckpt_vals["file"] = ""


        # Return training_state
        return training_state


    # Get training_state.json from the HF repo
    def get_training_state_from_hub(self, filename = "training_state.json"):

        # Let only rank 0 to run this (prevent mass API calls)
        if self.world_size > 1:
            self._smart_barrier("state_fetch_start")

        # Let rank == 0 load the .json
        if self.rank == 0:
            # Attempt to pull the training_state.json from hub
            try:
                # Try Downlaoding
                path = hf_hub_download(
                    repo_id=self.ckpt_config.model_repo_id,
                    filename = filename,
                    repo_type = "model",
                    token = self.ckpt_config.hf_token
                )

                with open(path, "r") as f:
                    training_state = json.load(f)

                # Successfully loaded
                print(f"✅ {filename} loaded successfully from {self.ckpt_config.model_repo_id}")
                training_state = self._deletion_status_updates(training_state)

            # If the repo doesn't exist, make the repo
            except RepositoryNotFoundError:
                # Print Error Statements
                print(f"⚠️ Repo: \"{self.ckpt_config.model_repo_id}\" was not found")
                print(f"🏗️ Creating Repo: {self.ckpt_config.model_repo_id}")

                # Create Repo
                create_repo(
                    repo_id = self.ckpt_config.model_repo_id,
                    token = self.ckpt_config.hf_token,
                    repo_type = "model",
                    private = False,
                    exist_ok = False,
                )

                # Make new training_state dict
                training_state = self._init_new_training_state()

            # The repo exists, but it's empty or doesn't have the state file yet
            except EntryNotFoundError:
                # Print info
                print(f"⚠️ {self.ckpt_config.model_repo_id} exists, but no {filename} found. Starting fresh.")
                training_state = self._init_new_training_state()

            # Unknown Error
            except Exception as e:
                # Catch-all for network timeouts, corrupted JSON, etc.
                print(f"❌ An unexpected error occurred: {e}. Starting from token zero.")
                training_state = self._init_new_training_state()

            # Dump the training_state from rank 0 into .json
            with open("local_training_state.json", "w") as f:
                json.dump(training_state, f)

        # Once rank 0 finishes, end the barrier
        if self.world_size > 1:
            self._smart_barrier("state_fetch_end")

        # Then every rank (including) loads the dict from "local_training_state.json"
        with open("local_training_state.json", "r") as f:
            final_state_dict = json.load(f)

        # Wait for EVERY rank to finish reading the file
        if self.world_size > 1:
            self._smart_barrier("state_read_complete")

        # Then Delete
        if self.rank == 0:
            import os
            if os.path.exists("local_training_state.json"):
                os.remove("local_training_state.json")

        # All Return the same dict
        return final_state_dict

    # Check to see if a checkpoint should be uploaded
    def check_upload_condition(self, curr_global_step):
        # Subtract offset if start_from_global (it treated step 0 = last checkpoint's step value)
        if not self.ckpt_config.start_from_global:
            curr_global_step -= self.actual_resume_step
        if curr_global_step <=0:
            return False

        # Save which interval we will use to calculate if we need to upload
        active_interval = None

        # Iterate through the sorted keys
        for threshold in sorted(self.ckpt_config.interval_dict.keys()):
            # If our curr_global_step is bigger than threshold, save it's value
            if curr_global_step >= threshold:
                active_interval = self.ckpt_config.interval_dict[threshold]
            else:
                # else we break since we haven't to this threshold yet
                break

        # If dictionary was empty (no checkpointing)
        if active_interval is None:
            return False

        # return whether the current step is a perfect multiple of the active_interval
        return (curr_global_step % active_interval == 0)

    # Resume Training: resume from the correct checkpoint
    # Pass the model and optimizer by reference to be initialized
    # Returns:
    # Checkpoint Entry Dictionary Snapshot if available
    # Dicionary with a bunch of 0s if all checkpoints were deleted or starting fresh or the actual snapshot dictionary
    # the actual resume step and session number
    def resume_training(self, model, optimizer):
        # Lazy Load torch to get correct version
        import torch

        # Keep track of all the valid steps
        valid_steps = []
        for step, data in self.training_state["checkpoints"].items():
            if data["status"] != "deleted":
                valid_steps.append(int(step))

        # Print messege and return 0 if its brand new
        if not valid_steps:
            if self.rank == 0:
                print("According to the training_state, every single checkpoint is invalid or deleted. Starting from ground 0")
            # return 0,0
            self.actual_resume_step = 0
            return {
                "hardware": self.hw_config.hardware_string,
                "curriculum_level": 0,
                "rows_processed_at_curr_level": 0,
                "total_tokens_processed_global": 0,
                "total_rows_processed_global": 0,
                "run_id": wandb.util.generate_id() if self.ckpt_config.use_wandb else "",
                "parquet_index": 1,
                "total_rows_processed_parquet": 0
            }, 0, 0, None

        # Get Actual valid resume step (e.g. I deleted the most recent version but it still says otherwise)
        actual_resume_step = max(valid_steps)
        self.actual_resume_step = actual_resume_step
        ckpt_entry = self.training_state["checkpoints"][str(actual_resume_step)]
        filename = ckpt_entry["file"]

        # Barrier
        if self.world_size > 1:
            self._smart_barrier("weight_download_start")

        # Only let rank 0 start downloading (the others will download from the runtime local disk):
        if self.rank == 0:
            print(f"Downloading {filename} from Hub...")
            try:
                hf_hub_download(
                    repo_id = self.ckpt_config.model_repo_id,
                    filename = filename,
                    repo_type = "model",
                    token = self.ckpt_config.hf_token,
                    local_dir = "."
                )
            except Exception as e:
                raise RuntimeError(f"Critical HF Download Failure for {filename}: {e}")

        # Barrier
        if self.world_size > 1:
            self._smart_barrier("weight_download_end")

        # Now that the model has been downloaded onto the runtime local disk, let each device download it
        # All ranks load the weights from the local file
        try:

            # Load the checkpoint
            pt_path = os.path.join(".", filename)
            ckpt = torch.load(pt_path, map_location='cpu', weights_only=False)

            # Get the model state
            model_state = ckpt['model_state']

            # GPUs and TPUs might add module. or not have it at all
            # Add or subtract this to maintain hardware compatibility
            new_state_dict = {}
            for k, v in model_state.items():
                if k.startswith('module.') and not hasattr(model, 'module'):
                    new_state_dict[k[7:]] = v
                elif not k.startswith('module.') and hasattr(model, 'module'):
                    new_state_dict[f'module.{k}'] = v
                else:
                    new_state_dict[k] = v

            # Load the model into dictionary
            model.load_state_dict(new_state_dict, strict=False)

            # Load the optmizer
            if optimizer and 'optimizer_state' in ckpt:
                optimizer.load_state_dict(ckpt['optimizer_state'])

            # Get scheduler state
            scheduler_state = ckpt.get('scheduler_state', None)

            # print success
            if self.rank == 0:
                print(f"Successfully loaded model and optimizer from Step {actual_resume_step}!")

            # Update the training_state session num
            self.training_state["session"] +=1

            # Sync up
            if self.world_size > 1:
                self._smart_barrier("model_optimizer_loaded")

            # Let rank = 0 delete the last checkpoint to resume
            if self.rank == 0:
                try:
                    os.remove(pt_path)
                except Exception as e:
                    printf("Loaded the model, but couldn't deleted intial checkpoint")

            # Just return the ckpt_entry; Extract the values later
            return ckpt_entry, actual_resume_step, self.training_state["session"], scheduler_state

        except Exception as e:
            raise RuntimeError(f"Critical Weight Loading Failure: {e}")

    # Saves the model to a .pt file
    # Makes checkpoint entry for training_state
    def save_checkpoint(self,
                        model,
                        optimizer,
                        scheduler,
                        global_step: int,
                        hardware_string: str,
                        metrics: dict,
                        is_tpu: bool,
                        curriculum_level: int,
                        total_tokens_processed_global: int,
                        total_rows_processed_global: int,
                        rows_processed_at_curr_level: int,
                        parquet_index: int,
                        total_rows_processed_parquet: int,
                        run_id = ""
                        ):

        # Lazy Load Torch
        import torch

        # Save filename
        step_str = str(global_step)
        filename = f"checkpoint-{global_step:06d}.pt"

        # If more than 1 worker, start barrier
        if self.world_size > 1:
            self._smart_barrier("save_start")

        if self.rank == 0:
            print(f"Saving model weights to {filename}...")

        # Ensure to use module or not to ensure compatibility
        save_dict = {
            "model_state" : model.module.state_dict() if hasattr(model, "module") else model.state_dict(),
            "optimizer_state" : optimizer.state_dict(),
            "scheduler_state" : scheduler.state_dict()
        }

        # Save using TPU or GPU .save()
        if is_tpu:
            import torch_xla.core.xla_model as xm
            xm.save(save_dict, filename)
        else:
            torch.save(save_dict, filename)

        # If more than 1 worker, end barrier
        if self.world_size > 1:
            self._smart_barrier("save_weights_end")

        # Update training_state (add checkpoint entry + update metadata)
        # Only let rank 0 change the state of the UPLOAD_REQUEST.json to ping the sidecar
        if self.rank == 0:

            # Ensure that the all latest tags get removed
            for step, data in self.training_state['checkpoints'].items():
                if data["status"] == "latest":
                    data["status"] = "history"

            # Create new checkpoint entry
            self.training_state["checkpoints"][step_str] = {
                "status": "latest",
                "file": filename,
                "hardware": hardware_string,
                "curriculum_level": curriculum_level,
                "rows_processed_at_curr_level": rows_processed_at_curr_level,
                "total_rows_processed_global": total_rows_processed_global,
                "total_tokens_processed_global": total_tokens_processed_global,
                "metrics": metrics,
                "run_id": run_id,
                "parquet_index": parquet_index,
                "total_rows_processed_parquet": total_rows_processed_parquet
            }

            # Ping sidecar by updating UPLOAD_REQUEST.json
            request_data = {
                "file_to_upload": filename,
                "step": global_step,
                "training_state_snapshot": self.training_state
            }

            with open(f"UPLOAD_REQUEST_{global_step}.json.tmp", "w") as f:
                json.dump(request_data, f)
            os.rename(f"UPLOAD_REQUEST_{global_step}.json.tmp", f"UPLOAD_REQUEST_{global_step}.json")

            # Print some bs idk lol
            print(f"Saved weights to local disk + updated training_state.json. Pinging Sidecar for Step {global_step}")

        # If more than 1 worker make sure other ranks wait for rank 0
        if self.world_size > 1:
            self._smart_barrier("save_training_state_end")



# Define Custom Learning Rate Scheduler
def get_curr_scheduler(optimizer, total_curr_level_steps, curr_max_lr, curr_min_lr, base_lr, warmup_steps = 0):

    # When given a lr_lambda function, the function multiplies this value by the base_lr
    # We want the real curr_max_lr / curr_min_lr, but we must divide before to cancel the multiplication
    max_mult = curr_max_lr / base_lr
    min_mult = curr_min_lr / base_lr

    def lr_lambda(current_step):
        # Warmup (only for first phase)
        if current_step < warmup_steps:
            return min_mult + (max_mult - min_mult) * (current_step / max(1, warmup_steps))

        # progress is a number between 0-1
        progress = (current_step - warmup_steps) / max (1, total_curr_level_steps - warmup_steps)

        # Make sure it doesn't go beyond 1
        progress = min (1.0, progress)

        # prog(0) = max_mult, prog(1) = min_mult
        return min_mult + 0.5 * (max_mult - min_mult) * (1 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda)



# Driver to Log Telemtry to WandB
class TelemetryDriver:

    # Initialize Hardware
    def __init__(self, rank, run_id, model_config, ckpt_config: CheckpointConfig, resume_step = 0, global_tokens_processed = 0):

        # Get Rank
        self.rank = rank

        # Get Checkpoint
        self.ckpt_config = ckpt_config

        # Get modeL_config
        self.model_config = model_config

        # Save new run object
        self.run = None

        # Save run type
        # "init"
        # "resume_from"
        # "fork_from"
        self.run_type = None

        # Initialize and resume data logging
        if self.rank == 0 and ckpt_config.use_wandb:

            # Starting Fresh
            if (resume_step == 0):
                self.run = wandb.init(
                    entity = ckpt_config.wandb_entity,
                    project = ckpt_config.wandb_project,
                    name = ckpt_config.wandb_name,
                    id = run_id,
                    config = vars(model_config)
                )
                self.run_type = "init"

            # If continuing and resume_from_work
            elif DOES_RESUME_FROM_WORK:
                self.run = wandb.init(
                        entity = ckpt_config.wandb_entity,
                        project = ckpt_config.wandb_project,
                        name = ckpt_config.wandb_name,
                        id = run_id,
                        resume_from = f"{run_id}?_step={resume_step}",
                        config = vars(model_config)
                )
                self.run_type = "resume_from"

            # Else use fork_from for better organization
            else:
                self.run = wandb.init(
                        entity = ckpt_config.wandb_entity,
                        project = ckpt_config.wandb_project,
                        name = ckpt_config.wandb_name,
                        fork_from = f"{run_id}?_step={resume_step}",
                        config = vars(model_config)
                )
                self.run_type = "fork_from"

    def _tele_key(self, k):
        # "layer_3_sqk_mean" -> "layer_3/sqk_mean" ; "lm_head_sz_mean" -> "lm_head/sz_mean"
        k = re.sub(r"^(layer_\d+)_", r"\1/", k)
        k = re.sub(r"^(lm_head)_",   r"\1/", k)
        return k

    def _make_heatmap(self, rows, label, cmap, global_step):
        matrix = np.stack(rows)
        fig, ax = plt.subplots(figsize=(10, 8))
        cax = ax.matshow(matrix, cmap=cmap, vmin=0.0, vmax=1.0)
        fig.colorbar(cax, label=label)
        ax.set_xlabel("Elastic Head Index"); ax.set_ylabel("Layer")
        ax.set_title(f"{label} (Step {global_step})")
        ax.set_yticks(range(matrix.shape[0]))
        img = wandb.Image(fig); plt.close(fig)
        return img

    def log_step(self, telemetry_dict, ce_loss, margin_loss, total_loss, global_step,
                 is_train=True, global_tokens_processed=None, easiness_mean=None):
        if self.rank != 0 or not self.ckpt_config.use_wandb or self.run is None:
            return

        prefix = "train" if is_train else "validation"
        log_payload = {
            f"{prefix}/ce_loss": ce_loss,
            f"{prefix}/margin_loss": margin_loss,
            f"{prefix}/total_loss": total_loss,
        }
        if easiness_mean is not None:
            # Training-only teacher telemetry. Never consumed by the model/router.
            log_payload[f"{prefix}/easiness_mean_telemetry_only"] = easiness_mean

        activation_rows, surrogate_rows = {}, {}

        for key, val in telemetry_dict.items():
            # Build compact per-layer heatmaps from fixed-shape tensors.
            m_hm = re.match(r"layer_(\d+)_hard_mask$", key)
            m_ss = re.match(r"layer_(\d+)_surrogate_scores$", key)
            if m_hm:
                activation_rows[int(m_hm.group(1))] = val.mean(dim=0).squeeze().numpy()
                continue
            if m_ss:
                surrogate_rows[int(m_ss.group(1))] = val.mean(dim=0).squeeze().numpy()
                log_payload[self._tele_key(key) + "_hist"] = wandb.Histogram(val.numpy())
                continue

            if isinstance(val, torch.Tensor):
                v = val.detach().cpu()
                log_payload[self._tele_key(key)] = (
                    wandb.Histogram(v.numpy()) if v.numel() > 1 else v.item()
                )
            elif isinstance(val, (int, float)):
                log_payload[self._tele_key(key)] = val

        if activation_rows:
            rows = [activation_rows[i] for i in sorted(activation_rows)]
            log_payload["router/activation_heatmap"] = self._make_heatmap(
                rows, "Elastic Activation Frequency", "cool", global_step
            )
        if surrogate_rows:
            rows = [surrogate_rows[i] for i in sorted(surrogate_rows)]
            log_payload["router/surrogate_heatmap"] = self._make_heatmap(
                rows, "STE Sigmoid Surrogate", "winter", global_step
            )

        # Useful architecture-wide aggregates.
        head_count_keys = [k for k in telemetry_dict if k.endswith("_total_head_count_mean")]
        law_keys = [k for k in telemetry_dict if k.endswith("_min_law_trigger_rate")]
        if head_count_keys:
            log_payload["router/mean_total_heads_all_layers"] = float(np.mean([
                telemetry_dict[k] for k in head_count_keys
            ]))
        if law_keys:
            log_payload["router/mean_min_law_trigger_rate"] = float(np.mean([
                telemetry_dict[k] for k in law_keys
            ]))

        log_payload["global_tokens_processed"] = global_tokens_processed
        wandb.log(log_payload, step=global_step)



def print_phase13_router_console(telemetry_dict, global_step):
    """Rank-0 compact diagnostics; called only on existing telemetry steps."""
    counts = []
    triggers = []
    shadow_triggers = []
    thresholds = []
    stages = []
    ramps = []
    for i in range(12):
        ck = f"layer_{i}_total_head_count_mean"
        tk = f"layer_{i}_min_law_trigger_rate"
        stk = f"layer_{i}_shadow_min_law_trigger_rate"
        bk = f"layer_{i}_global_threshold"
        sk = f"layer_{i}_routing_stage"
        rk = f"layer_{i}_margin_ramp"
        if ck in telemetry_dict:
            counts.append(float(telemetry_dict[ck]))
        if tk in telemetry_dict:
            triggers.append(float(telemetry_dict[tk]))
        if stk in telemetry_dict:
            shadow_triggers.append(float(telemetry_dict[stk]))
        if bk in telemetry_dict:
            thresholds.append(float(telemetry_dict[bk]))
        if sk in telemetry_dict:
            stages.append(int(round(float(telemetry_dict[sk]))))
        if rk in telemetry_dict:
            ramps.append(float(telemetry_dict[rk]))
    if counts:
        stage_id = stages[0] if stages else -1
        stage_name = {0: "DENSE", 1: "CALIBRATE-16", 2: "ELASTIC"}.get(stage_id, "?")
        print(
            f"Phase13 Router @ {global_step} | stage={stage_name} "
            f"| total heads mean={np.mean(counts):.2f} "
            f"| layer range=[{min(counts):.2f},{max(counts):.2f}] "
            f"| min-law={np.mean(triggers):.3f} "
            f"| shadow-min={np.mean(shadow_triggers):.3f} "
            f"| margin-ramp={np.mean(ramps):.2f} "
            f"| global-threshold={np.mean(thresholds):+.3f}"
        )


# Function that all devices will run (ran from the launch function right above)
def train_worker(rank, hw_config, data_config, ckpt_config):

    # We need each TPU process to communicate to the main process
    # The best and cheapest way is to create a thread that write a file onto disk
    # This sits outside the training loop so we can detect whether the training loop is hanging
    import threading

    # Start Daemon Thread to write
    heartbeat_stop = threading.Event()

    # Fucntion to write the time
    # If the entire train_worker process dies, this thread dies and fails to write
    # This is how we will detect changes
    def heartbeat_report():
        path = f"/tmp/heartbeat_rank_{rank}.txt"
        # True if even got set, false if timeout expired
        while not heartbeat_stop.wait(timeout = 10):
            try:
                with open(path, "w") as f:
                    f.write(str(time.time()))
            except Exception:
                pass # Ensure training doesn't crash because of this john

    # Start the thread
    heartbeat_thread = threading.Thread(target=heartbeat_report, daemon = True)
    heartbeat_thread.start()

    # Smart Barrier (Rendezvous) to prevent data races & ensure all devices make it to certain step
    def _smart_barrier(self, name="barrier"):
        if hw_config.world_size <= 1:
            return  # No synchronization needed for single device

        if hw_config.device_type == "tpu":
            import torch_xla.core.xla_model as xm
            xm.rendezvous(name)
        elif hw_config.device_type == "cuda":
            import torch.distributed as dist
            if dist.is_initialized():
                dist.barrier()

    # Lazy Load
    import sys
    import traceback
    import os # Add os

    if hw_config.hf_token:
        os.environ["HF_TOKEN"] = hw_config.hf_token

    try:
        # Lazy Load
        import datasets
        datasets.config.TF_AVAILABLE = False
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from transformers import AutoTokenizer
        import SpanMLMCollatorWithEasiness
        from model import HELMConfig, HELMForMaskedLM


        # Default for Data Collator
        is_tpu = False

        # Load correct packages and get device
        if hw_config.device_type == "tpu":
            # Lazy Load even more for TPU
            import torch_xla.core.xla_model as xm
            import torch_xla.distributed.parallel_loader as pl
            import torch_xla.runtime as xr

            device = xm.xla_device()
            is_tpu = True

            # get real world size just in case
            real_world_size = xr.world_size()
            if hw_config.world_size != real_world_size:
                if rank == 0:
                    print(f"⚠️ CONFIG MISMATCH: Adjusting world size to {real_world_size}")
                hw_config.world_size = real_world_size

        elif hw_config.device_type == "cuda":
            # Set cuda device to torch
            torch.cuda.set_device(rank)
            device = torch.device(f"cuda:{rank}")
            # Initialize Distributed comm framework
            # acts like a rendezvous
            # NVIDIA Collective Communications Library (nccl)
            if hw_config.world_size > 1:
                import torch.distributed as dist
                dist.init_process_group("nccl", rank=rank, world_size=hw_config.world_size)

        else:
            # Default to CPU just in case
            device = torch.device("cpu")


        if rank == 0:
            print(f"Rank 0 is online: {device}.")

        # Define Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            data_config.tokenizer_name, token=hw_config.hf_token
        )

        # Define MLMDataStrategy
        data_strat = MLMDataStrategy(
            rank = rank, world_size = hw_config.world_size, is_tpu = is_tpu,config = data_config, hf_token=hw_config.hf_token
        )

        # Define Checkpoint Driver
        checkpoint_driver = CheckpointDriver(
            hw_config = hw_config, data_config = data_config, ckpt_config = ckpt_config,
            rank = rank, world_size = hw_config.world_size
        )

        # Get the total steps of the data
        num_rows_dict = checkpoint_driver.training_state["total_rows_dict"]
        dataset_total_steps = num_rows_dict["all"] // hw_config.target_gbs

        # Phase13A does NOT feed externally labeled easiness into the router.
        # Keep the dataset column for telemetry, but construct the model without any
        # easiness CDF / target-count dependency.
        helm_config = HELMConfig(
            vocab_size=len(tokenizer),
            pad_token_id=tokenizer.pad_token_id,
            dataset_total_steps=dataset_total_steps,
        )

        if rank == 0:
            print(
                "🔥 Phase13A-v2 router schedule: "
                f"dense=[0,{helm_config.dense_router_warmup_steps}) | "
                f"calibrate-16=[{helm_config.dense_router_warmup_steps},{helm_config.elastic_start_step}) | "
                f"elastic={helm_config.elastic_start_step}+ | "
                f"controller/margin ramp={helm_config.elastic_transition_steps} steps"
            )

        # Create model and attach to device
        model = HELMForMaskedLM(helm_config).to(device)

        # Require DDP to Wrap the model if using cuda
        if hw_config.device_type == "cuda" and hw_config.world_size > 1:
            from torch.nn.parallel import DistributedDataParallel as DDP
            # model = DDP(model, device_ids=[rank], find_unused_parameters=True)
            # There shouldn't be extra args
            model = DDP(model, device_ids=[rank])

        # Define Optimizer
        optimizer = optim.AdamW(model.parameters(), lr = helm_config.base_lr, weight_decay = helm_config.weight_decay)

        # Zero the gradient
        optimizer.zero_grad()

        # Define CE Loss
        loss_fct = nn.CrossEntropyLoss()
        loss_fct_sum = nn.CrossEntropyLoss(reduction="sum")  # ignore_index defaults to -100
        def chunked_ce(logits, labels, vocab_size, n_chunks=8):
            flat_logits = logits.reshape(-1, vocab_size)
            flat_labels = labels.reshape(-1)
            total = flat_logits.size(0)
            chunk = (total + n_chunks - 1) // n_chunks
            valid = (flat_labels != loss_fct.ignore_index).sum().clamp_min(1)
            loss_sum = flat_logits.new_zeros(())
            for i in range(0, total, chunk):
                loss_sum = loss_sum + loss_fct_sum(
                    flat_logits[i:i + chunk].float(),
                    flat_labels[i:i + chunk],
                )
            return loss_sum / valid

        # Set data type that will be used
        dtype = hw_config.dtype

        # Allow Scaler
        use_scaler = hw_config.use_scaler
        scaler = torch.amp.GradScaler('cuda') if hw_config.device_type == "cuda" and use_scaler else None

        # ========== CHECKPOINT TECHNOLOGICA ==========

        # Loading the model/optimizer returns the most recent, valid / undeleted checkpoint
        ckpt_snapshot, actual_resume_step, session_number, scheduler_state = checkpoint_driver.resume_training(model, optimizer)

        # Extract the values from the ckpt_snapshot
        start_curr_level = ckpt_snapshot["curriculum_level"]
        rows_processed_at_curr_level = ckpt_snapshot["rows_processed_at_curr_level"]
        total_rows_processed_global = ckpt_snapshot["total_rows_processed_global"]
        total_tokens_processed_global = ckpt_snapshot["total_tokens_processed_global"]
        parquet_index = ckpt_snapshot["parquet_index"]
        total_rows_processed_parquet = ckpt_snapshot["total_rows_processed_parquet"]

        # Set the global step to where we left off from the previous checkpoint
        global_step = actual_resume_step

        # If starting fresh initialize weights
        if actual_resume_step == 0:
            # Use .module to access the original HELMForMaskedLM if wrapped in DDP
            unwrapped_model = model.module if hasattr(model, "module") else model
            unwrapped_model.apply(unwrapped_model._init_weights)

        # Extract run_id (for wandb logging)
        run_id = ckpt_snapshot["run_id"]

        # If we aren't using resume_from, have incremental session number names
        if not DOES_RESUME_FROM_WORK:
            ckpt_config.wandb_name = f"{ckpt_config.wandb_name}-{session_number:05}"

        # Initialize TelemetryDriver
        telemetry_driver = TelemetryDriver(
            rank = rank,
            run_id = run_id,
            ckpt_config = ckpt_config,
            model_config = helm_config,
            resume_step = actual_resume_step
        )

        # ========== CURRICULUM LOOP ==========
        # Curriculum Outer Loop (starting from the current curriculum):
        for level in range(start_curr_level,len(data_config.curriculum_subset_names)):

            # --- ADDED PARQUET STOP LOGIC ---
            if parquet_index >= data_config.parquet_stop_index:
                if rank == 0:
                    print(f"🛑 Reached parquet stop index ({data_config.parquet_stop_index}). Stopping curriculum.")
                break
            # --------------------------------

            total_curr_level_steps = checkpoint_driver.training_state["total_rows_dict"][str(level)] // hw_config.target_gbs

            # Reset optimizer's internal LR (each curr_level turns it -> min_lr, so reset is required)
            if not (scheduler_state is not None and level == start_curr_level):
                for param_group in optimizer.param_groups:
                    param_group['lr'] = helm_config.base_lr

            if level == 0:
                warmup_steps = int(total_curr_level_steps * 0.01)
                scheduler = get_curr_scheduler(
                    optimizer, total_curr_level_steps, helm_config.base_lr,
                    helm_config.min_lr, helm_config.base_lr, warmup_steps
                )
            elif level == 1:
                scheduler = get_curr_scheduler(
                    optimizer, total_curr_level_steps, helm_config.base_lr * 0.35,
                    helm_config.min_lr, helm_config.base_lr, 0
                )
            elif level == 2:
                scheduler = get_curr_scheduler(
                    optimizer, total_curr_level_steps, helm_config.base_lr * 0.18,
                    helm_config.min_lr, helm_config.base_lr, 0
                )

            # If resuming, overwrite the scheduler's internal state
            # (restores step counter so cosine decay continues from where it left off)
            if scheduler_state is not None and level == start_curr_level:
                scheduler.load_state_dict(scheduler_state)
                scheduler_state = None


            # Sync up devices
            if hw_config.world_size > 1:
                _smart_barrier("load_scheduler")

            # Set Model to Training Mode
            model.train()

            # Get Profile Level
            level_profile = hw_config.hardware_profile[level]

            # Get micro batch size (mb) and use gradient checkpointing (use_ckpt)
            hw_config.batch_size = level_profile["mb"]

            # CRITICAL: Unwrap the model first to handle DDP (GPUs) vs Raw (TPUs)
            unwrap_model = model.module if hasattr(model, "module") else model
            # Change the Model Configs using the safely unwrapped model
            unwrap_model.config.use_ckpt = level_profile["use_ckpt"]
            unwrap_model.model.use_ckpt = level_profile["use_ckpt"] # HELMModel caches this

            # Save seq_len somewhere just in case if we need to use it
            seq_len = level_profile["sl"]

            # Calculate grad_accum_steps
            hw_config.grad_accum_steps = max(1, hw_config.target_gbs // (hw_config.batch_size * hw_config.world_size))

            # Define the Collator
            collator = SpanMLMCollatorWithEasiness.SpanMLMCollatorWithEasiness(
                config = data_config, tokenizer = tokenizer
            )

            validation_file_path = ""
            if rank == 0:
                # Load Validation parquet for current curriculum level
                validation_file_path, val_parquet_num_rows, parquet_curr_level = data_strat.download_parquet(is_train = False, index = parquet_index)

            if hw_config.world_size > 1:
                _smart_barrier("start_download_validation")

            if rank !=0:
                # Load Validation parquet for current curriculum level
                validation_file_path, val_parquet_num_rows, parquet_curr_level = data_strat.download_parquet(is_train = False, index = parquet_index) # , loaded_parquet_file_path = validation_file_path)



            # Define validation_dataloader
            validation_loader = data_strat.get_mlm_data_loader(
                parquet_file_path = validation_file_path,
                collate_fn = collator,
                batch_size = hw_config.batch_size,
                parquet_index = parquet_index,
                is_train = False,
            )

            # var holding a new train_file_path so it can be preloaded without training hiccups
            # This shouldn't affect the curriculum level, I'm just storing it for consistency
            new_train_file_path = ""
            new_train_parquet_num_rows = 0
            new_parquet_curr_level = 0

            # if TPU is being used, apply the ParallelLoader().per_device_loader()
            if is_tpu:
                validation_loader = pl.ParallelLoader(validation_loader, [device]).per_device_loader(device)



            # ========== TRAIN_LOADER PREPPER LOOP ==========

            while True:

                # --- ADDED PARQUET STOP LOGIC ---
                if parquet_index >= data_config.parquet_stop_index:
                    if rank == 0:
                        print(f"🛑 Reached parquet stop index ({data_config.parquet_stop_index}). Stopping data loader loop.")
                    break
                # --------------------------------

                # Load Validation parquet for current curriculum level unless it's been preloaded
                if new_train_file_path == "":
                    train_file_path = ""
                    if rank == 0:
                        train_file_path, train_parquet_num_rows, parquet_curr_level = data_strat.download_parquet(is_train = True, index = parquet_index)

                    if hw_config.world_size > 1:
                        _smart_barrier("start_download_training")

                    if rank !=0:
                        train_file_path, train_parquet_num_rows, parquet_curr_level = data_strat.download_parquet(is_train = True, index = parquet_index) # , loaded_parquet_file_path = train_file_path)

                else:
                    train_file_path = new_train_file_path
                    train_parquet_num_rows = new_train_parquet_num_rows
                    parquet_curr_level = new_parquet_curr_level
                    new_train_file_path = ""
                    new_train_parquet_num_rows = 0
                    new_parquet_curr_level = 0

                # If we were on the last curriculum level's parquet and downloaded the next, break out
                if (parquet_curr_level != level):
                    break

                # Define train dataloader
                train_loader = data_strat.get_mlm_data_loader(
                    parquet_file_path = train_file_path,
                    collate_fn = collator,
                    skip_rows =  total_rows_processed_parquet,
                    batch_size = hw_config.batch_size,
                    parquet_index = parquet_index,
                    is_train = True,
                )

                # if TPU is being used, apply the ParallelLoader().per_device_loader()
                if is_tpu:
                    train_loader = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)

                # ========== TRAINING LOOP ==========
                # Fixed-shape controller accumulators. These are reset after every
                # optimizer step; no variable-length routing tensors are created.
                controller_usage_accum = torch.zeros(
                    (helm_config.num_hidden_layers,
                     helm_config.num_attention_heads - helm_config.num_permanent_heads),
                    device=device, dtype=torch.float32
                )
                controller_count_accum = torch.zeros(
                    (helm_config.num_hidden_layers,), device=device, dtype=torch.float32
                )
                controller_cutoff_accum = torch.zeros(
                    (helm_config.num_hidden_layers,), device=device, dtype=torch.float32
                )

                # Loop through each batch
                for step, batch in enumerate(train_loader):

                    # If SHUTDOWN_FILE exists, set the break boolean and break
                    # Claude recommends to call the checkpoint driver, but then we might train on the same information
                    # Therefore, we will just set the flag and dip
                    if os.path.exists(SHUTDOWN_FILE):
                        if rank == 0:
                            print("SHUTDOWN_FILE is up. Ending...")
                        break

                    # Get Batch's input ids, labels, and attn_mask (we don't have one but just in case) and attach it to device
                    input_ids = batch["input_ids"].to(device)
                    labels = batch["labels"].to(device)
                    attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)

                    # Phase13A: easiness is telemetry only. It is NEVER passed to model().
                    easiness_score = batch.get("easiness_score", None)

                    # TPU/GPU forward is shape-static. The router internally uses only
                    # fixed [B,E] tensor operations (cosine, comparisons, TopK, scatter, where).
                    if hw_config.device_type == "cuda":
                        with torch.autocast(device_type="cuda", dtype=dtype):
                            logits, margin_loss = model(
                                input_ids=input_ids,
                                attention_mask=attention_mask,
                                current_step=global_step,
                            )
                            ce_loss = chunked_ce(logits, labels, helm_config.vocab_size)
                            total_loss = (ce_loss + margin_loss) / hw_config.grad_accum_steps
                    else:
                        with torch.autocast(device_type="xla", dtype=torch.bfloat16):
                            logits, margin_loss = model(
                                input_ids=input_ids,
                                attention_mask=attention_mask,
                                current_step=global_step,
                            )
                            ce_loss = chunked_ce(logits, labels, helm_config.vocab_size)
                            total_loss = (ce_loss + margin_loss) / hw_config.grad_accum_steps

                    # Collect fixed-shape hard-routing observations BEFORE backward.
                    # They are detached and are not controller mutations. Accumulation
                    # spans the same microbatches as the optimizer gradient accumulation.
                    unwrapped_model = model.module if hasattr(model, "module") else model
                    usage_obs, count_obs, cutoff_obs = unwrapped_model.get_router_observation_means()
                    controller_usage_accum = controller_usage_accum + usage_obs
                    controller_count_accum = controller_count_accum + count_obs
                    controller_cutoff_accum = controller_cutoff_accum + cutoff_obs


                    if scaler is not None:
                        scaler.scale(total_loss).backward()
                    else:
                        total_loss.backward()
                    # Mark every micro batch to prevent accumulating the entire graph
                    if is_tpu:
                        xm.mark_step()

                    # Once Gradient has been accumulated, step the model and the optimizer
                    if (step + 1) % hw_config.grad_accum_steps == 0:

                        # Apply Gradient Clipping Here instead of inside the model
                        # Start by unwrapping model form DDP or not
                        unwrapped_model = model.module if hasattr(model, "module") else model

                        # Now apply gradient clipping here to multi-view router's learnable params
                        # (skip cleanly for the no-router baseline, which has no mlt_vw_rtr)
                        if hw_config.device_type == "tpu" or hw_config.device_type == "cuda":
                            for block in unwrapped_model.model.blocks:
                                if hasattr(block, "mlt_vw_rtr"):
                                    torch.nn.utils.clip_grad_value_(
                                        block.mlt_vw_rtr.parameters(),
                                        clip_value = helm_config.router_grad_clip
                                    )

                        if is_tpu:
                            xm.optimizer_step(optimizer)
                            xm.mark_step()
                        elif scaler is not None:
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            optimizer.step()

                        scheduler.step()

                        # Normalize nGPT/cosine-direction matrices.
                        unwrapped_model.normalize_ngpt_matrices()

                        # ---------------- LOSS-FREE ROUTER CONTROLLER ----------------
                        # Average observations across grad-accum microbatches first.
                        controller_usage_mean = controller_usage_accum / float(hw_config.grad_accum_steps)
                        controller_count_mean = controller_count_accum / float(hw_config.grad_accum_steps)
                        controller_cutoff_mean = controller_cutoff_accum / float(hw_config.grad_accum_steps)

                        # Then average the SAME fixed-size tensors across replicas.
                        # PyTorch/XLA all_reduce is an in-place collective; shapes stay
                        # [layers, elastic_heads] and [layers] on every step.
                        if is_tpu and hw_config.world_size > 1:
                            xm.all_reduce(
                                xm.REDUCE_SUM,
                                [controller_usage_mean, controller_count_mean, controller_cutoff_mean],
                                scale=1.0 / float(hw_config.world_size),
                            )
                        elif hw_config.device_type == "cuda" and hw_config.world_size > 1:
                            import torch.distributed as dist
                            dist.all_reduce(controller_usage_mean, op=dist.ReduceOp.SUM)
                            dist.all_reduce(controller_count_mean, op=dist.ReduceOp.SUM)
                            dist.all_reduce(controller_cutoff_mean, op=dist.ReduceOp.SUM)
                            controller_usage_mean.div_(float(hw_config.world_size))
                            controller_count_mean.div_(float(hw_config.world_size))
                            controller_cutoff_mean.div_(float(hw_config.world_size))

                        # Controller buffers are NOT optimizer Parameters. Every replica
                        # receives identical global observations and performs the same
                        # deterministic update.
                        unwrapped_model.update_router_controllers(
                            controller_usage_mean,
                            controller_count_mean,
                            controller_cutoff_mean,
                            global_step,
                        )

                        # Flush the small fixed-shape controller graph before next batch.
                        if is_tpu:
                            xm.mark_step()

                        # Reset fixed-shape observation accumulators for next optimizer step.
                        controller_usage_accum = torch.zeros_like(controller_usage_accum)
                        controller_count_accum = torch.zeros_like(controller_count_accum)
                        controller_cutoff_accum = torch.zeros_like(controller_cutoff_accum)

                        # Zero the gradient
                        optimizer.zero_grad()
                        global_step += 1


                        # Calculating the values for the save_checkpoint
                        # Should just be the target gbs, but just in case
                        rows_this_step = hw_config.batch_size * hw_config.grad_accum_steps * hw_config.world_size
                        tokens_this_step = rows_this_step * seq_len

                        # Increment the total amount of rows processed in parquet
                        total_rows_processed_parquet += rows_this_step

                        # Preload the next parquet and save vars if the current parquet is 95% done
                        # Maybe make this asynchronous ???
                        if (((float) (total_rows_processed_parquet) / train_parquet_num_rows) >= .95) and new_train_file_path == "":
                            new_train_file_path, new_train_parquet_num_rows, new_parquet_curr_level = data_strat.download_parquet(is_train = True, index = parquet_index + 1)


                        rows_processed_at_curr_level +=  rows_this_step
                        total_rows_processed_global += rows_this_step
                        total_tokens_processed_global += tokens_this_step

                        report_total_loss = to_float(ce_loss) + to_float(margin_loss)

                        # Log Data to Wandb
                        if global_step < 100 or global_step % 10 == 0:

                            # Save telemetry_dict
                            telemetry_dict = unwrapped_model.get_telemetry()

                            easiness_mean = (
                                to_float(easiness_score.float().mean())
                                if easiness_score is not None else None
                            )
                            telemetry_driver.log_step(
                                telemetry_dict=telemetry_dict,
                                ce_loss=to_float(ce_loss),
                                margin_loss=to_float(margin_loss),
                                total_loss=report_total_loss,
                                global_step=global_step,
                                is_train=True,
                                global_tokens_processed=total_tokens_processed_global,
                                easiness_mean=easiness_mean,
                            )
                            if rank == 0:
                                print_phase13_router_console(telemetry_dict, global_step)


                        # Use 1 device (rank = 0) to calculate the real loss
                        if rank == 0:
                            print(f"Step {global_step} | Total Loss: {report_total_loss:.4f} | CE: {to_float(ce_loss):.4f} | Margin: {to_float(margin_loss):.5f}")


                        # Save the model if the time is right (based on interval_dict from CheckpoingConfig)
                        if checkpoint_driver.check_upload_condition(global_step):

                            # Ensure correct run_id is saved (should only change when using fork_from)
                            if telemetry_driver.run_type == "fork_from":
                                run_id = telemetry_driver.run.id

                            checkpoint_driver.save_checkpoint(
                                model = model,
                                optimizer = optimizer,
                                scheduler = scheduler,
                                global_step = global_step,
                                hardware_string = hw_config.hardware_string,
                                metrics = {
                                "Total Loss": round(report_total_loss, 5),
                                    "CE Loss": round(to_float(ce_loss), 5),
                                    "Margin Loss": round(to_float(margin_loss), 6)

                                },
                                is_tpu = is_tpu,
                                curriculum_level = level,
                                total_tokens_processed_global = total_tokens_processed_global,
                                total_rows_processed_global = total_rows_processed_global,
                                rows_processed_at_curr_level = rows_processed_at_curr_level,
                                parquet_index = parquet_index,
                                total_rows_processed_parquet = total_rows_processed_parquet,
                                run_id = run_id,
                            )

                        # ========== VALIDATION LOOP ==========
                        # Log Valdiation every 500 steps
                        if global_step % (500 if not TESTING_MODE else 50) == 0:

                            if rank == 0:
                                print("⏳ Calculating Validation...")

                            # zero the gradient again just in case
                            optimizer.zero_grad()

                            # Put model into eval mode
                            model.eval()

                            # Initialize accumulators
                            total_val_loss = 0.0
                            total_ce_loss = 0.0
                            total_margin_loss = 0.0

                            # Loop through the validation_loader
                            for step, batch in enumerate(validation_loader):

                                if step > hw_config.validation_step_num:
                                    break

                                # Get Batch's input ids, labels, and attn_mask and attach it to device
                                input_ids = batch["input_ids"].to(device)
                                labels = batch["labels"].to(device)
                                attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)

                                # Use no_grad to prevent OOM during validation
                                with torch.no_grad():
                                    # GPUs require Autocast for Mixed Precision. TPUs handle it natively via Env Variables.
                                    if hw_config.device_type == "cuda":
                                        with torch.autocast(device_type="cuda", dtype=dtype):
                                            logits, margin_loss = model(input_ids=input_ids, attention_mask=attention_mask, current_step=global_step)
                                            ce_loss = chunked_ce(logits, labels, helm_config.vocab_size)
                                            val_loss = ce_loss + margin_loss
                                    else:
                                        logits, margin_loss = model(input_ids=input_ids, attention_mask=attention_mask, current_step=global_step)
                                        ce_loss = chunked_ce(logits, labels, helm_config.vocab_size)
                                        val_loss = ce_loss + margin_loss

                                # Add Loss Values
                                total_val_loss += to_float(val_loss)
                                total_ce_loss += to_float(ce_loss)
                                total_margin_loss += to_float(margin_loss)

                                if rank == 0 and step % 10 == 0:
                                    print(f"Completed Validation Step {step}/{hw_config.validation_step_num} - we are alive")


                            # Calculate and print the final averages once the loop naturally finishes
                            if hw_config.validation_step_num > 0:
                                avg_val_loss = total_val_loss / hw_config.validation_step_num
                                avg_ce_loss = total_ce_loss / hw_config.validation_step_num
                                avg_margin_loss = total_margin_loss / hw_config.validation_step_num

                                # Normalize the model's weights
                                unwrapped_model = model.module if hasattr(model, "module") else model
                                unwrapped_model.normalize_ngpt_matrices()

                                # Save telemetry_dict
                                telemetry_dict = unwrapped_model.get_telemetry()

                                # Log the Data to WandB
                                telemetry_driver.log_step(
                                    telemetry_dict=telemetry_dict,
                                    ce_loss=avg_ce_loss,
                                    margin_loss=avg_margin_loss,
                                    total_loss=avg_val_loss,
                                    global_step=global_step,
                                    is_train=False,
                                )

                                if rank == 0:
                                    print(f"Total Loss: {avg_val_loss:.4f} | CE: {avg_ce_loss:.4f} | Margin: {avg_margin_loss:.5f}")

                            # Call model.train
                            model.train()

                # break out of parquet loop
                if os.path.exists(SHUTDOWN_FILE):
                    break

                if rank == 0:
                    print(f"📦 Finished parquet {parquet_index} (level {level}). Advancing.")

                # Delete the consumed parquet so disk doesn't fill up
                data_strat.delete_parquet(train_file_path)

                # Advance to the next parquet and reset within-parquet row counter
                parquet_index += 1
                total_rows_processed_parquet = 0

            # Delete valdiation parquet once the curriculum is over
            data_strat.delete_parquet(validation_file_path)

            # # break out of curriculum loop
            if os.path.exists(SHUTDOWN_FILE):
                break

        # Destroy once all of these johns are done
        if hw_config.device_type == "cuda" and hw_config.world_size > 1:
            dist.destroy_process_group()

        # Stop the heartbeat and delete (so when rerun / revive occurs, it doesn't use the old file)
        heartbeat_stop.set()

    except Exception as e:
        print(f"\n❌ FATAL WORKER ERROR ON RANK {rank}:")
        traceback.print_exc()
        return



def sidecar_uploader_loop(hf_token, repo_id):
    # LAZY LOAD
    import os
    import json
    import time
    import signal
    from datetime import datetime, timezone
    from huggingface_hub import HfApi

    # Ignore stop signals
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    signal.signal(signal.SIGTERM, signal.SIG_IGN)

    # Get HF API Token to upload
    api = HfApi(token=hf_token)

    # Forever Loop to constantly check
    while True:

        # Check to see if any valid upload requests exist & take the step size
        upload_requests = []
        for file in os.listdir("."):
            if file.startswith("UPLOAD_REQUEST_") and file.endswith(".json"):
                upload_requests.append(int(file.replace("UPLOAD_REQUEST_", "").replace(".json", "")))

        # Sort the list and take the first request
        if upload_requests:

            # Get the next upload_request and process that first
            next_upload = sorted(upload_requests)[0]
            upload_request_filename = f"UPLOAD_REQUEST_{next_upload}.json"

            # Try to upload the model and training_state.json to HF HUB
            try:

                # Open the UPLOADER_REQUEST.json
                with open(upload_request_filename, "r") as f:
                    UPLOAD_REQUEST = json.load(f)

                # Get filename and the step
                model_filename = UPLOAD_REQUEST["file_to_upload"]
                step = UPLOAD_REQUEST["step"]
                training_state_snapshot = UPLOAD_REQUEST["training_state_snapshot"]

                # Print Messeage
                print(f"⏳ Attempting to upload {model_filename} to {repo_id}")

                # Upload the model first (most unstable action to do before uplaoding the .json)
                if os.path.exists(model_filename):
                    api.upload_file(
                        path_or_fileobj=model_filename,
                        path_in_repo=model_filename,
                        repo_id=repo_id,
                        repo_type="model"
                    )
                else:
                    print(f"❌ Failed to Upload. {upload_request_filename} was pinged, but {model_filename} does not exist")

                # Format the .json to include whitespace
                formatted_json_str = json.dumps(training_state_snapshot, indent=4)
                json_bytes = formatted_json_str.encode('utf-8')
                fileobj = io.BytesIO(json_bytes)

                # Upload the training_state.json
                api.upload_file(
                    path_or_fileobj=fileobj,
                    path_in_repo="training_state.json",
                    repo_id=repo_id,
                    repo_type="model"
                )


                # Delete the big .pt file immediately to free disk space
                os.remove(model_filename)
                os.remove(upload_request_filename)

                # Squash history in background — don't block the next upload
                try:
                    api.super_squash_history(repo_id=repo_id)
                except Exception:
                    pass  # Non-critical, repo just gets bigger

                print(f"✅ Successfully uploaded {model_filename} @ step {step} to {repo_id}")

            except Exception as e:
                print(f"❌ Failed to upload to HF: {e}")
                print("Trying again in 5 seconds...")

        # Pause 5 seconds before rechecking if UPLOAD_REQUEST.json exists
        time.sleep(5)



if __name__ == "__main__":

    # Allow to kill all processes / end them correctly --------
    import signal
    import sys
    import threading
    import shutil

    # Shutdown Event (Essentially Thread-safe boolean)
    shutdown_event = threading.Event()
    shutdown_count = {"n": 0}

    # Shutdown manager
    # This is called once automatically. Press again if the shutdown is
    # completely cooked, skipping all cleanup.
    def graceful_shutdown(signum, frame):
        shutdown_count["n"] +=1
        if shutdown_count["n"] >=2:
            # Hard termination (x2 hits)
            print(f"2nd signal {signum} received. Hard Exit")
            try:
                with open(USER_STOP_MARKER, "w") as f:
                    f.write("hard")
            except Exception:
                pass
            os._exit(1)
        else:
            # Graceful termination
            print(f"Termination Signal ({signum}). Sending shutdown file to workers... (Ctrl+C again to hard-exit)")
            try:
                with open(SHUTDOWN_FILE, "w") as f:
                    f.write("user")
                with open(USER_STOP_MARKER, "w") as f:
                    f.write("graceful")
            except Exception:
                pass

            # Mark the sutdown event to gracefully shutdown
            shutdown_event.set()
            # raise KeyboardInterrupt


    # Catch Kaggle's Stop button (SIGTERM) and Keyboard Interrupts (SIGINT)
    signal.signal(signal.SIGTERM, graceful_shutdown)
    signal.signal(signal.SIGINT, graceful_shutdown)
    # ---------------------------------------------------------

    # Delete all existing SHUTDOWN_FILE and USER_STOP_MARKER that could've been from previous runs
    for f in [SHUTDOWN_FILE, USER_STOP_MARKER]:
        if os.path.exists(f):
            try:
                os.remove(f)
            except Exception:
                pass

    # Ensure environment is primed for TPU PJRT
    for key in ["XRT_TPU_CONFIG", "PJRT_SELECT_DEVICE", "TPU_PROCESS_ADDRESSES"]:
        os.environ.pop(key, None)
    os.environ["PJRT_DEVICE"] = "TPU"

    # Set wandb key to None
    wandb_key = None

    # Get dummy ckpt_config to check if use_wandb is true:
    dummy_ckpt_cfg = CheckpointConfig()

    # Get wandb api key if wandb is being used
    if dummy_ckpt_cfg.use_wandb:
        wandb_key = get_secret("WANDB_API_KEY")

    # HF Token (must)
    hf_token = get_secret("HF_TOKEN")

    # Initialize all configs
    HW_CFG = HardwareConfig(hf_token = hf_token)
    DATA_CFG = MLMDataConfig()
    CKPT_CFG = None

    if not TESTING_MODE:
        CKPT_CFG = CheckpointConfig(hf_token = hf_token, wandb_key = wandb_key if wandb_key is not None else "")
    else:
        CKPT_CFG = CheckpointConfig(hf_token = hf_token, wandb_key = wandb_key if wandb_key is not None else "", interval_dict = {0:10})


    # Log into wandb if we are logging in:
    if wandb_key and CKPT_CFG.use_wandb:
        os.environ["WANDB_API_KEY"] = wandb_key
        wandb.login()
    elif not CKPT_CFG.use_wandb:
        print("wandb disabled. Change use_wandb in the CheckpointConfig to True if you intended to log")
    else:
        print("⚠️ wandb_key was NULL. Make sure you allow secrets on Colab or Kaggle. Continuing Anonymous Logging")


    # Use the 'spawn' context to prevent C++ state corruption
    ctx = multiprocessing.get_context('spawn')

    # Update: Sidecar respawn. Insteaf spawning one time, create respawning thread

    # Create holder so watchdog can swap the sidecar
    sidecar_holder = {"proc": None}

    def spawn_sidecar():
        # Loading Sidecar via isolated CPU thread to upload
        uploader_process = ctx.Process(
            target = sidecar_uploader_loop,
            args = (CKPT_CFG.hf_token, CKPT_CFG.model_repo_id),
            daemon = False
        )
        uploader_process.start()
        return uploader_process

    sidecar_holder["proc"] = spawn_sidecar()

    # Watchdog worker function to respawn sidecar
    def sidecar_watchdog():
        crash_count = 0
        MAX_CRASHES = 5
        # Checks every 30 seconds
        while not shutdown_event.wait(timeout = 30):
            proc = sidecar_holder["proc"]
            # is_alive() is really waitpid(pid, WNOHANG)
            # True when running
            # False when zombie
            if not proc.is_alive():
                # Reap the zombie
                proc.join(timeout = 5)
                if (crash_count >= MAX_CRASHES):
                    print(f"Sidecar crashed {crash_count} times. No reboot card for you anymore...")
                    return
                crash_count +=1
                exitcode = proc.exitcode
                print(f"⚠️ Sidecar died (exitcode={exitcode}, crash #{crash_count}). Respawning...")
                sidecar_holder["proc"] = spawn_sidecar()

    # Create Watchdog
    sidecar_watchdog_thread = threading.Thread(target=sidecar_watchdog, daemon = True)
    sidecar_watchdog_thread.start()

    # Simple training watchdog — warn on stale heartbeats, that's it.
    # With JAX_PLATFORMS=cpu, the C++ runtime handles cleanup.
    # If a worker hangs, press Stop twice → os._exit(1).
    def training_watchdog():
        STALE_WARN = 120
        warned = set()
        while not shutdown_event.wait(timeout=30):
            now = time.time()
            for r in range(HW_CFG.world_size):
                path = f"/tmp/heartbeat_rank_{r}.txt"
                if not os.path.exists(path):
                    continue
                try:
                    with open(path) as f:
                        last = float(f.read().strip())
                except Exception:
                    continue
                age = now - last
                if age > STALE_WARN and r not in warned:
                    print(f"⚠️ WATCHDOG: Rank {r} heartbeat {age:.0f}s old. Possible hang.")
                    warned.add(r)
                elif age <= STALE_WARN and r in warned:
                    print(f"✅ WATCHDOG: Rank {r} recovered.")
                    warned.discard(r)

    # Start trainer watchdog for all threads
    training_watchdog_thread = threading.Thread(target=training_watchdog, daemon=True)
    training_watchdog_thread.start()


    # Prepare Hardware Driver
    driver = HardwareDriver(HW_CFG, DATA_CFG, CKPT_CFG)

    # Try to launch training process
    try:
        driver.launch(train_worker)
    except KeyboardInterrupt:
        print("⚠️ Training interrupted by signal or pause button")
    except Exception as e:
        print(f"💀 Summ done messed up cuh {e}")
        import traceback
        traceback.print_exc()


    finally:
        print("🧹 Cleanup starting...")
        shutdown_event.set()

        # Wait for sidecar to finish any in-flight upload
        time_count = 0
        MAX_DRAIN_SEC = 600
        while time_count < MAX_DRAIN_SEC:
            still_uploading = any(
                file.startswith("UPLOAD_REQUEST_") and file.endswith(".json")
                for file in os.listdir(".")
            )
            if not still_uploading:
                break
            if not sidecar_holder["proc"].is_alive():
                print("⚠️ Sidecar died before finishing uploads.")
                break
            if time_count % 30 == 0:
                print(f"⏳ Sidecar uploading... ({time_count}s elapsed)")
            time.sleep(5)
            time_count += 5
        if time_count >= MAX_DRAIN_SEC:
            print(f"⚠️ Sidecar drain timed out after {MAX_DRAIN_SEC}s.")

        # Kill sidecar
        try:
            sidecar_holder["proc"].kill()
            sidecar_holder["proc"].join(timeout=10)
        except Exception:
            pass

        # Clean up TPU lockfile (safety net)
        if HW_CFG.device_type == "tpu":
            if os.path.exists("/tmp/libtpu_lockfile"):
                try:
                    os.remove("/tmp/libtpu_lockfile")
                    print("🧹 Removed stale libtpu_lockfile")
                except Exception:
                    pass

        # Clean heartbeat files
        for i in range(HW_CFG.world_size):
            try:
                os.remove(f"/tmp/heartbeat_rank_{i}.txt")
            except Exception:
                pass

        # Clean shutdown files
        for f in [SHUTDOWN_FILE, USER_STOP_MARKER]:
            try:
                os.remove(f)
            except Exception:
                pass

        print("💅 Okay girl... shutdown is  ✨✨COMPLETE✨✨")

Overwriting parallel_hardware_trainer.py


In [5]:
%%writefile SpanMLMCollatorWithEasiness.py
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dataclasses import dataclass
from transformers import AutoTokenizer
import json
import multiprocessing

class ConfigJson():
    def __init__(self, **kwargs):
        # Assign attributes from keyword arguments
        for key, value in kwargs.items():
            setattr(self, key, value)

    @classmethod
    def from_json(cls, json_path):
        with open(json_path, "r") as file:
            data = json.load(file)
        return cls(**data)



class SpanMLMCollatorWithEasiness:

    # Define Init
    # Most of the things are just stored in the config lwk
    # Imma just pull from here
    def __init__(self, config = None, tokenizer = None, mlm_probability = None, mlm_use_span_masking = None, mlm_span_length = None, use_easiness = True):
        
        # Defaults
        self.mlm_probability = 0.15
        self.mlm_use_span_masking = False
        self.mlm_span_length = 3
        self.tokenizer = None

        # Override with Config (if provided)
        if (config is not None):
            if (isinstance(config, str)):
                try:
                    config = ConfigJson.from_json(config)
                except Exception as e:
                    raise ValueError(f"Blud was not a json. Either some sort of dictionary ahhh config or .json. Error: {e}")

            self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
            self.mlm_probability = config.mlm_probability
            self.mlm_use_span_masking = config.mlm_use_span_masking
            self.mlm_span_length = config.mlm_span_length

        if tokenizer is not None:
            self.tokenizer = tokenizer
        if mlm_probability is not None:
            self.mlm_probability = mlm_probability
        if mlm_use_span_masking is not None:
            self.mlm_use_span_masking = mlm_use_span_masking
        if mlm_span_length is not None:
            self.mlm_span_length = mlm_span_length

            
        if self.tokenizer is None:
            raise ValueError("Tokenizer is None. Either pass in the config or tokenizer")
        
        self.use_easiness = use_easiness

        # AI HELP:
        # Create vocab for legal words to replace
        valid_ids = [i for i in range(len(self.tokenizer)) if i not in self.tokenizer.all_special_ids]
        self.valid_vocab = torch.tensor(valid_ids)

    # __call__() function returning a DataLoader with Span_masking
    def __call__(self, data):

        # Convert List of Dictionaries into 1 large tensor
        # inside, convert extract input_ids from dictionary
        batch_input_ids = [d["input_ids"] for d in data]
        input_ids = torch.tensor(batch_input_ids)
        labels = input_ids.clone()
        easiness_score = torch.tensor([d["easiness_score"] for d in data], dtype=torch.float32)

        # Set mlm_prob
        mlm_prob = self.mlm_probability
        if (self.mlm_use_span_masking):
            mlm_prob /= self.mlm_span_length

        # Create the Masking Tensor
        masked_tensor = torch.full(input_ids.shape, mlm_prob, dtype=torch.float32)



        # =====================================================================
        # THE GUARD STEP: Protect special tokens from being masked
        # =====================================================================
        # 1. Ask the tokenizer which tokens in each row are special (CLS, SEP, PAD)
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(seq, already_has_special_tokens=True) 
            for seq in input_ids.tolist()
        ]
        
        # 2. Convert that nested list into a PyTorch Boolean tensor
        special_tokens_map = torch.tensor(special_tokens_mask, dtype=torch.bool)
        
        # 3. Overwrite the probability in masked_tensor to 0.0 wherever special_tokens_map is True
        masked_tensor.masked_fill_(special_tokens_map, value=0.0)
        # =====================================================================


        # Apply Bernoulli to get 1s and 0s for the masks
        masked_tensor = torch.bernoulli(masked_tensor).bool()

        temp_clone = masked_tensor.clone()
        # Apply rolling for span masking
        if (self.mlm_use_span_masking):
            for i in range (1, self.mlm_span_length):
                rolled_tensor = torch.roll(temp_clone, shifts = i)
                rolled_tensor[:,:i] = False
                masked_tensor = masked_tensor | rolled_tensor

        # Mask all untampered tokens with -100
        labels[~masked_tensor] = -100
        
        # Create new Tensor w/ random values from 0-1
        type_tensor = torch.rand(input_ids.shape)

        # Create 80% mask_token_tensor
        mask_token_tensor = (type_tensor <= .8) & masked_tensor

        # Create 10% corrupted_token_tensor
        corrupted_token_tensor = (type_tensor > .8) & (type_tensor <= .9) & masked_tensor

        # Apply Mask tokens to input_ids
        input_ids[mask_token_tensor] = self.tokenizer.mask_token_id

        # LOOK HERE ##################################################
        
        # Apply corrupted tokens to input_ids
        num_to_replace = corrupted_token_tensor.sum().item()

        # Generate random indices for the valid_vocab_tensor
        indices = torch.randint(0,len(self.valid_vocab), (num_to_replace,))

        # Take the words form valid_vocab
        random_words = self.valid_vocab[indices]

        # Apply the words to the input_ids
        input_ids[corrupted_token_tensor] = random_words

        # ############################################################
        
        # Return Dictionary (based on if easiness is requested)

        if self.use_easiness:
            return {"input_ids": input_ids, "labels": labels, "easiness_score": easiness_score}

        return {"input_ids": input_ids, "labels": labels}

Writing SpanMLMCollatorWithEasiness.py


In [13]:
import subprocess, os, signal, time, sys

SHUTDOWN_FILE = "/tmp/SHUTDOWN_REQUESTED"
USER_STOP_MARKER = "/tmp/USER_STOPPED_TRAINING"
MAX_RETRIES = 5
BASE_BACKOFF = 30
MIN_RUNTIME = 60

# Clean stale files
for f in [SHUTDOWN_FILE, USER_STOP_MARKER, "/tmp/libtpu_lockfile"]:
    try: os.remove(f)
    except: pass
for i in range(8):
    try: os.remove(f"/tmp/heartbeat_rank_{i}.txt")
    except: pass

attempt = 0
while attempt < MAX_RETRIES:
    attempt += 1
    print(f"\n{'='*40}")
    print(f"🚀 Launch attempt {attempt}/{MAX_RETRIES}")
    print(f"{'='*40}\n")

    # Isolate training in its own session
    proc = subprocess.Popen(
        ["python", "-u", "parallel_hardware_trainer.py"],
        start_new_session=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0  # unbuffered binary
    )

    start = time.time()
    user_stopped = False

    try:
        # Read output line-by-line, decode to string, and flush to the notebook
        for line in iter(proc.stdout.readline, b''):
            sys.stdout.write(line.decode('utf-8', errors='replace'))
            sys.stdout.flush()
        proc.wait()

    except KeyboardInterrupt:
        # Stop button pressed — write shutdown file, wait for clean exit
        user_stopped = True
        print("\n🛑 Stop pressed. Writing shutdown file...")
        with open(SHUTDOWN_FILE, "w") as f:
            f.write("user")
        with open(USER_STOP_MARKER, "w") as f:
            f.write("graceful")

        # Drain remaining output while waiting for exit
        print("⏳ Waiting for clean exit (up to 5 min)...")
        try:
            remaining = proc.stdout.read()
            if remaining:
                sys.stdout.write(remaining.decode('utf-8', errors='replace'))
                sys.stdout.flush()
            proc.wait(timeout=300)
            print(f"✅ Exited cleanly (code {proc.returncode}).")
        except KeyboardInterrupt:
            print("⚠️ Second stop — force killing.")
            try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print("⚠️ Timed out — force killing.")
            try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait(timeout=10)

    runtime = time.time() - start

    # User pressed stop → don't restart
    if user_stopped or os.path.exists(USER_STOP_MARKER):
        print("🛑 User-initiated stop. Not restarting.")
        try: os.remove(USER_STOP_MARKER)
        except: pass
        try: os.remove(SHUTDOWN_FILE)
        except: pass
        break

    # Clean exit → training finished
    if proc.returncode == 0:
        print(f"✅ Training finished after {runtime:.0f}s.")
        break

    # Crash → retry with backoff
    print(f"💥 Crashed (code {proc.returncode}) after {runtime:.0f}s.")

    if attempt >= MAX_RETRIES:
        print(f"⚠️ Hit max retries ({MAX_RETRIES}). Giving up.")
        break

    # Fast crash = exponential backoff, long run = short backoff
    if runtime < MIN_RUNTIME:
        backoff = BASE_BACKOFF * (2 ** (attempt - 1))
        print(f"⚠️ Fast crash — backing off {backoff}s...")
    else:
        backoff = BASE_BACKOFF
        print(f"⏳ Restarting in {backoff}s (resume from last checkpoint)...")

    # Clean stale files before retry
    for f in [SHUTDOWN_FILE, USER_STOP_MARKER, "/tmp/libtpu_lockfile"]:
        try: os.remove(f)
        except: pass

    time.sleep(backoff)

print("\n🏁 Launcher done.")


🚀 Launch attempt 1/5

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jhui16 (jhui16-university-of-maryland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jhui16 (jhui16-university-of-maryland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260810_013638-dm7i3bui
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run phase7a-warmup-v2-00000
wandb: ⭐️ View project at https://wandb.ai/jhui16-university-of-maryland/HELM-v1-10B-Run
wandb: 🚀 View run at https://wandb.ai/jhui16-university-of-maryland/HELM-v1-10B-Run/runs/dm7i3bui
/usr/local/lib/python3.12/site-packages/huggingfa